# Milestone 4 — Domain-aware harness development

## Relationship-aware schema grounding

The bare schema baseline required the model to infer how tables connect merely
from similarly named columns. GPT-5.2 handled this reliably, but Ministral-3B
frequently invented direct relationships such as `account.client_id` and
`loan.client_id`.

This stage adds only structural information:

- tables and columns;
- declared types;
- primary keys;
- relationships between foreign-key-like columns and primary keys;
- safe quoting of SQLite identifiers.

It does not add:

- column descriptions;
- categorical values;
- sample rows;
- business rules;
- few-shot examples;
- benchmark-derived hints;
- generated-SQL repair.

The immediate goal is improved schema grounding and executable-SQL rate, not
complete domain understanding.

In [1]:
from collections import defaultdict
from pathlib import Path
import sqlite3


db_path = Path("../data/financial.sqlite").resolve()

conn = sqlite3.connect(
    f"file:{db_path.as_posix()}?mode=ro",
    uri=True,
)


def quote_identifier(identifier: str) -> str:
    """Safely quote a SQLite identifier."""
    return '"' + identifier.replace('"', '""') + '"'


table_names = [
    row[0]
    for row in conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        ORDER BY name
        """
    )
]

print("Tables:", table_names)

Tables: ['account', 'card', 'client', 'disp', 'district', 'loan', 'order', 'trans']


In [2]:
columns_by_table = {}
primary_keys_by_table = {}
declared_relationships = []

for table_name in table_names:
    quoted_table = quote_identifier(table_name)

    columns = conn.execute(
        f"PRAGMA table_info({quoted_table})"
    ).fetchall()

    columns_by_table[table_name] = columns

    primary_keys_by_table[table_name] = [
        column[1]
        for column in sorted(
            columns,
            key=lambda column: column[5],
        )
        if column[5] > 0
    ]

    foreign_keys = conn.execute(
        f"PRAGMA foreign_key_list({quoted_table})"
    ).fetchall()

    for foreign_key in foreign_keys:
        declared_relationships.append(
            (
                table_name,
                foreign_key[3],  # child column
                foreign_key[2],  # parent table
                foreign_key[4],  # parent column
            )
        )

for table_name in table_names:
    print("=" * 60)
    print("Table:", table_name)
    print(
        "Primary key:",
        primary_keys_by_table[table_name] or "not declared",
    )

print("=" * 60)
print("Declared relationships:", declared_relationships)

Table: account
Primary key: ['account_id']
Table: card
Primary key: ['card_id']
Table: client
Primary key: ['client_id']
Table: disp
Primary key: ['disp_id']
Table: district
Primary key: ['district_id']
Table: loan
Primary key: ['loan_id']
Table: order
Primary key: ['order_id']
Table: trans
Primary key: ['trans_id']
Declared relationships: [('account', 'district_id', 'district', 'district_id'), ('card', 'disp_id', 'disp', 'disp_id'), ('client', 'district_id', 'district', 'district_id'), ('disp', 'client_id', 'client', 'client_id'), ('disp', 'account_id', 'account', 'account_id'), ('loan', 'account_id', 'account', 'account_id'), ('order', 'account_id', 'account', 'account_id'), ('trans', 'account_id', 'account', 'account_id')]


In [3]:
primary_key_owners = defaultdict(list)

for table_name, primary_keys in primary_keys_by_table.items():
    for primary_key in primary_keys:
        primary_key_owners[primary_key].append(table_name)


inferred_relationships = []

for child_table, columns in columns_by_table.items():
    child_primary_keys = set(
        primary_keys_by_table[child_table]
    )

    for column in columns:
        column_name = column[1]

        if column_name in child_primary_keys:
            continue

        candidate_parent_tables = [
            parent_table
            for parent_table in primary_key_owners.get(
                column_name,
                [],
            )
            if parent_table != child_table
        ]

        # Infer only when there is one unambiguous PK owner.
        if len(candidate_parent_tables) == 1:
            inferred_relationships.append(
                (
                    child_table,
                    column_name,
                    candidate_parent_tables[0],
                    column_name,
                )
            )

inferred_relationships.sort()

print("Inferred relationship candidates:")

for (
    child_table,
    child_column,
    parent_table,
    parent_column,
) in inferred_relationships:
    print(
        f'  "{child_table}"."{child_column}"'
        f' -> "{parent_table}"."{parent_column}"'
    )

Inferred relationship candidates:
  "account"."district_id" -> "district"."district_id"
  "card"."disp_id" -> "disp"."disp_id"
  "client"."district_id" -> "district"."district_id"
  "disp"."account_id" -> "account"."account_id"
  "disp"."client_id" -> "client"."client_id"
  "loan"."account_id" -> "account"."account_id"
  "order"."account_id" -> "account"."account_id"
  "trans"."account_id" -> "account"."account_id"


In [4]:
print("Tables inspected:", len(table_names))
print(
    "Declared relationships:",
    len(declared_relationships),
)
print(
    "Inferred relationship candidates:",
    len(inferred_relationships),
)
print("Database rows or categorical values displayed: 0")
print("Model requests made: 0")

Tables inspected: 8
Declared relationships: 8
Inferred relationship candidates: 8
Database rows or categorical values displayed: 0
Model requests made: 0


## Render relationship-aware schema

In [5]:
assert set(declared_relationships) == set(
    inferred_relationships
)

print(
    "Declared and inferred relationship graphs match:",
    len(declared_relationships),
    "relationships",
)

Declared and inferred relationship graphs match: 8 relationships


In [ ]:
from hashlib import sha256

def build_relationship_schema_context(
    conn: sqlite3.Connection,
) -> str:
    """Render tables, primary keys, and declared relationships."""
    tables = [
        row[0]
        for row in conn.execute(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name
            """
        )
    ]

    context_lines = ["SQLite schema:"]

    all_relationships = []

    for table_name in tables:
        quoted_table = quote_identifier(table_name)

        columns = conn.execute(
            f"PRAGMA table_info({quoted_table})"
        ).fetchall()

        foreign_keys = conn.execute(
            f"PRAGMA foreign_key_list({quoted_table})"
        ).fetchall()

        entries = []

        for column in columns:
            column_name = column[1]
            declared_type = column[2] or "UNKNOWN"
            is_primary_key = column[5] > 0

            definition = (
                f"{quote_identifier(column_name)} "
                f"{declared_type}"
            )

            if is_primary_key:
                definition += " PRIMARY KEY"

            entries.append(definition)

        normalized_foreign_keys = sorted(
            (
                foreign_key[3],  # child column
                foreign_key[2],  # parent table
                foreign_key[4],  # parent column
            )
            for foreign_key in foreign_keys
        )

        for (
            child_column,
            parent_table,
            parent_column,
        ) in normalized_foreign_keys:
            entries.append(
                "FOREIGN KEY "
                f"({quote_identifier(child_column)}) "
                "REFERENCES "
                f"{quote_identifier(parent_table)} "
                f"({quote_identifier(parent_column)})"
            )

            all_relationships.append(
                (
                    table_name,
                    child_column,
                    parent_table,
                    parent_column,
                )
            )

        context_lines.append(
            f"\nCREATE TABLE {quoted_table} ("
        )

        for index, entry in enumerate(entries):
            comma = "," if index < len(entries) - 1 else ""
            context_lines.append(f"  {entry}{comma}")

        context_lines.append(");")

    # A compact second representation helps a small model find
    # the join path without changing the underlying information.
    context_lines.append("\nRelationships:")

    for (
        child_table,
        child_column,
        parent_table,
        parent_column,
    ) in sorted(all_relationships):
        context_lines.append(
            f"- {quote_identifier(child_table)}."
            f"{quote_identifier(child_column)} "
            "references "
            f"{quote_identifier(parent_table)}."
            f"{quote_identifier(parent_column)}"
        )

    return "\n".join(context_lines)


relationship_schema_context = (
    build_relationship_schema_context(conn)
)

relationship_schema_sha256 = sha256(
    relationship_schema_context.encode("utf-8")
).hexdigest()

print(relationship_schema_context)
print("\nRelationship-schema SHA-256:")
print(relationship_schema_sha256)

SQLite schema:

CREATE TABLE "account" (
  "account_id" INTEGER PRIMARY KEY,
  "district_id" INTEGER,
  "frequency" TEXT,
  "date" DATE,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "card" (
  "card_id" INTEGER PRIMARY KEY,
  "disp_id" INTEGER,
  "type" TEXT,
  "issued" DATE,
  FOREIGN KEY ("disp_id") REFERENCES "disp" ("disp_id")
);

CREATE TABLE "client" (
  "client_id" INTEGER PRIMARY KEY,
  "gender" TEXT,
  "birth_date" DATE,
  "district_id" INTEGER,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "disp" (
  "disp_id" INTEGER PRIMARY KEY,
  "client_id" INTEGER,
  "account_id" INTEGER,
  "type" TEXT,
  FOREIGN KEY ("account_id") REFERENCES "account" ("account_id"),
  FOREIGN KEY ("client_id") REFERENCES "client" ("client_id")
);

CREATE TABLE "district" (
  "district_id" INTEGER PRIMARY KEY,
  "A2" TEXT,
  "A3" TEXT,
  "A4" TEXT,
  "A5" TEXT,
  "A6" TEXT,
  "A7" TEXT,
  "A8" INTEGER,
  "A9" INTEGER,
  "A10"

In [7]:
import json
import os

from azure.identity import (
    AzureCliCredential,
    get_bearer_token_provider,
)
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from domain_adapt.evaluation import compare_sql_results
from domain_adapt.finch import make_reference_compatible
from domain_adapt.frontier import load_baseline_config
from domain_adapt.slm import request_sql as request_slm_sql


load_dotenv(find_dotenv(usecwd=True))

dataset_path = Path(
    "../data/finch_dataset.json"
).resolve()
manifest_path = Path(
    "../configs/eval_manifest_v1.json"
).resolve()
slm_config_path = Path(
    "../configs/slm_baseline_v1.json"
).resolve()

records = json.loads(
    dataset_path.read_text(encoding="utf-8")
)
manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)
slm_config = load_baseline_config(slm_config_path)

working_ids = set(
    manifest["selection"]["working_dev"]["ids"]
)
frozen_ids = set(
    manifest["selection"]["frozen_eval"]["ids"]
)

slm_deployment_name = os.environ[
    "AZURE_SLM_MODEL_DEPLOYMENT_NAME"
]

tenant_id = os.environ["AZURE_TENANT_ID"]
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")

if not endpoint.endswith("/openai/v1"):
    endpoint = f"{endpoint}/openai/v1"

credential = AzureCliCredential(tenant_id=tenant_id)

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint}/",
    api_key=token_provider,
)

In [8]:
question_id = 14

assert question_id in working_ids
assert question_id not in frozen_ids

matches = [
    row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
    and row["question_id"] == question_id
]

assert len(matches) == 1
example = matches[0]

print("Question ID:", example["question_id"])
print("Difficulty:", example["difficulty"])
print("Question:", example["question"])

Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD


In [10]:
response, generated_sql = request_slm_sql(
    client=client,
    deployment_name=slm_deployment_name,
    instructions=slm_config["instructions"],
    schema_context=relationship_schema_context,
    question=example["question"],
    max_tokens=slm_config["max_tokens"],
)

print("Response model:", response.model)
print("Finish reason:", response.choices[0].finish_reason)
print("\nRaw model output:")
print(generated_sql)
print("\nUsage:")
print(response.usage)

Response model: ministral-3b-2410
Finish reason: stop

Raw model output:
SELECT "account"."account_id"
FROM "account"
JOIN "trans" ON "account"."account_id" = "trans"."account_id"
WHERE "trans"."date" < '1997-01-01'
AND "trans"."amount" > 3000

Usage:
CompletionUsage(completion_tokens=67, prompt_tokens=837, total_tokens=904, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [11]:
compatible_reference_sql, applied_rules = (
    make_reference_compatible(example["SQL"])
)

comparison = compare_sql_results(
    conn=conn,
    reference_sql=compatible_reference_sql,
    candidate_sql=generated_sql,
)

print("Question ID:", question_id)
print("Reference-only compatibility rules:", applied_rules)
print("Passed:", comparison.equivalent)
print("Reason:", comparison.reason)
print("Expected row count:", comparison.expected_row_count)
print("Candidate row count:", comparison.candidate_row_count)

Question ID: 14
Reference-only compatibility rules: ['sqlite_four_digit_year']
Passed: False
Reason: unordered results differ
Expected row count: 3591
Candidate row count: 194721


## Measure all 12 working questions

In [12]:
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter


working_examples = sorted(
    [
        row
        for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in working_ids
    ],
    key=lambda row: row["question_id"],
)

assert len(working_examples) == 12

relationship_results = []

for index, working_example in enumerate(
    working_examples,
    start=1,
):
    started = perf_counter()

    try:
        response, generated_sql = request_slm_sql(
            client=client,
            deployment_name=slm_deployment_name,
            instructions=slm_config["instructions"],
            schema_context=relationship_schema_context,
            question=working_example["question"],
            max_tokens=slm_config["max_tokens"],
        )

        latency_seconds = perf_counter() - started

        compatible_reference_sql, applied_rules = (
            make_reference_compatible(working_example["SQL"])
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": applied_rules,
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": response.choices[0].finish_reason,
            "latency_seconds": round(latency_seconds, 3),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "latency_seconds": round(latency_seconds, 3),
            "usage": {},
            "api_error": f"{type(error).__name__}: {error}",
        }

    relationship_results.append(result)

    outcome = "PASS" if result["passed"] else "FAIL"

    print(
        f"[{index:02d}/{len(working_examples)}] "
        f"ID {result['question_id']:>3} "
        f"({result['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

[01/12] ID   5 (medium) FAIL: unordered results differ
[02/12] ID  11 (medium) FAIL: candidate execution failed: ambiguous column name: account_id
[03/12] ID  14 (easy  ) FAIL: column count differs: 1 != 2
[04/12] ID  16 (easy  ) PASS: unordered results match
[05/12] ID  45 (easy  ) FAIL: candidate execution failed: misuse of aggregate: MAX()
[06/12] ID  49 (medium) FAIL: unordered results differ
[07/12] ID  61 (hard  ) FAIL: candidate execution failed: no such column: a.type
[08/12] ID  69 (easy  ) FAIL: unordered results differ
[09/12] ID  81 (hard  ) FAIL: unordered results differ
[10/12] ID  89 (easy  ) FAIL: unordered results differ
[11/12] ID  99 (easy  ) FAIL: unordered results differ
[12/12] ID 105 (medium) FAIL: candidate execution failed: no such column: d.district_id


In [13]:
def is_executable(result: dict) -> bool:
    return (
        result["api_error"] is None
        and not result["reason"].startswith(
            "candidate execution failed"
        )
        and not result["reason"].startswith(
            "model request failed"
        )
    )


passed_count = sum(
    result["passed"]
    for result in relationship_results
)

executable_count = sum(
    is_executable(result)
    for result in relationship_results
)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in relationship_results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(result["passed"] for result in subset),
        "total": len(subset),
    }

failure_reasons = Counter(
    result["reason"]
    for result in relationship_results
    if not result["passed"]
)

total_usage = {
    "prompt_tokens": sum(
        result["usage"].get("prompt_tokens", 0)
        for result in relationship_results
    ),
    "completion_tokens": sum(
        result["usage"].get("completion_tokens", 0)
        for result in relationship_results
    ),
    "total_tokens": sum(
        result["usage"].get("total_tokens", 0)
        for result in relationship_results
    ),
}

print(
    "Accuracy:",
    f"{passed_count}/12",
    f"({passed_count / 12:.1%})",
)
print(
    "Executable:",
    f"{executable_count}/12",
    f"({executable_count / 12:.1%})",
)
print("By difficulty:", difficulty_summary)
print("Failure reasons:", dict(failure_reasons))
print("API errors:", sum(
    result["api_error"] is not None
    for result in relationship_results
))
print("Token usage:", total_usage)

Accuracy: 1/12 (8.3%)
Executable: 8/12 (66.7%)
By difficulty: {'easy': {'passed': 1, 'total': 6}, 'medium': {'passed': 0, 'total': 4}, 'hard': {'passed': 0, 'total': 2}}
Failure reasons: {'unordered results differ': 6, 'candidate execution failed: ambiguous column name: account_id': 1, 'column count differs: 1 != 2': 1, 'candidate execution failed: misuse of aggregate: MAX()': 1, 'candidate execution failed: no such column: a.type': 1, 'candidate execution failed: no such column: d.district_id': 1}
API errors: 0
Token usage: {'prompt_tokens': 10022, 'completion_tokens': 796, 'total_tokens': 10818}


In [14]:
relationship_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "benchmark": manifest["benchmark"],
    "split": "working_dev",
    "variant": "relationship-schema-v1",
    "deployment_name": slm_deployment_name,
    "catalog_model": slm_config["catalog_model"],
    "observed_response_model": (
        slm_config["observed_response_model"]
    ),
    "base_config_sha256": (
        "3f84c96358c50fce912ac642c2b3f38d"
        "3486f0da65f9e3ad9e52a6a7fd0ed762"
    ),
    "schema_sha256": relationship_schema_sha256,
    "declared_relationship_count": len(
        declared_relationships
    ),
    "generation": {
        "attempts_per_question": 1,
        "temperature": "provider_default",
        "candidate_sql_repair": False,
    },
    "summary": {
        "passed": passed_count,
        "total": 12,
        "accuracy": passed_count / 12,
        "executable": executable_count,
        "difficulty": difficulty_summary,
        "failure_reasons": dict(failure_reasons),
        "usage": total_usage,
    },
    "results": relationship_results,
}

relationship_run_path = Path(
    "../data/runs/"
    "ministral_3b_relationship_schema_working_dev_v1.json"
).resolve()

relationship_run_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

relationship_run_path.write_text(
    json.dumps(relationship_run, indent=2),
    encoding="utf-8",
)

print("Saved:", relationship_run_path)

Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_relationship_schema_working_dev_v1.json


In [15]:
baseline_run_path = Path(
    "../data/runs/"
    "ministral_3b_minimal_working_dev_v1.json"
).resolve()

baseline_run = json.loads(
    baseline_run_path.read_text(encoding="utf-8")
)

baseline_by_id = {
    result["question_id"]: result
    for result in baseline_run["results"]
}

relationship_by_id = {
    result["question_id"]: result
    for result in relationship_results
}

print(
    "ID | Base exec | Rel exec | Base pass | Rel pass"
)

for question_id in sorted(working_ids):
    baseline_result = baseline_by_id[question_id]
    relationship_result = relationship_by_id[question_id]

    print(
        f"{question_id:>3} | "
        f"{str(is_executable(baseline_result)):<9} | "
        f"{str(is_executable(relationship_result)):<8} | "
        f"{str(baseline_result['passed']):<9} | "
        f"{relationship_result['passed']}"
    )

ID | Base exec | Rel exec | Base pass | Rel pass
  5 | True      | True     | False     | False
 11 | False     | False    | False     | False
 14 | False     | True     | False     | False
 16 | True      | True     | True      | True
 45 | True      | False    | False     | False
 49 | False     | True     | False     | False
 61 | False     | False    | False     | False
 69 | True      | True     | False     | False
 81 | False     | True     | False     | False
 89 | False     | True     | False     | False
 99 | True      | True     | False     | False
105 | False     | False    | False     | False


## Discover authoritative column descriptions

In [16]:
from pathlib import Path
import csv

data_root = Path("../data")

description_csvs = sorted(
    path
    for path in data_root.rglob("*.csv")
    if "database_description" in {
        part.lower()
        for part in path.parts
    }
)

print("Description CSV files found:", len(description_csvs))

for path in description_csvs:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    print("-" * 80)
    print("Path:", path)
    print("Columns:", reader.fieldnames)
    print("Metadata rows:", len(rows))

Description CSV files found: 0


In [17]:
all_csvs = sorted(data_root.rglob("*.csv"))

print("All CSV files under data:", len(all_csvs))

for path in all_csvs:
    print(path)

All CSV files under data: 0


## Download metadata and inspect it

```
Invoke-WebRequest `
  -Uri 'https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip' `
  -OutFile '.\data\bird_dev.zip'

Expand-Archive `
  -Path '.\data\bird_dev.zip' `
  -DestinationPath '.\data\bird_dev_outer' `
  -Force

Expand-Archive `
  -Path '.\data\bird_dev_outer\dev_20240627\dev_databases.zip' `
  -DestinationPath '.\data\bird_dev_databases' `
  -Force

Get-ChildItem `
  -Path '.\data\bird_dev_databases' `
  -Directory `
  -Recurse |
Where-Object {
  $_.Name -eq 'database_description' -and
  $_.Parent.Name -eq 'financial'
} |
Select-Object FullName


In [18]:
from pathlib import Path
import csv
import hashlib

metadata_root = Path(
    "../data/bird_dev_databases/dev_databases/"
    "financial/database_description"
)

description_csvs = sorted(metadata_root.glob("*.csv"))

print("Metadata directory:", metadata_root.resolve())
print("CSV count:", len(description_csvs))
print("Files:", [path.name for path in description_csvs])

metadata_hash = hashlib.sha256()

for path in description_csvs:
    raw_bytes = path.read_bytes()

    metadata_hash.update(path.name.encode("utf-8"))
    metadata_hash.update(b"\0")
    metadata_hash.update(raw_bytes)
    metadata_hash.update(b"\0")

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    print("-" * 80)
    print("File:", path.name)
    print("Columns:", reader.fieldnames)
    print("Metadata rows:", len(rows))

print("-" * 80)
print("Financial metadata SHA-256:", metadata_hash.hexdigest())

Metadata directory: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\bird_dev_databases\dev_databases\financial\database_description
CSV count: 8
Files: ['account.csv', 'card.csv', 'client.csv', 'disp.csv', 'district.csv', 'loan.csv', 'order.csv', 'trans.csv']
--------------------------------------------------------------------------------
File: account.csv
Columns: ['original_column_name', 'column_name', 'column_description', 'data_format', 'value_description', '']
Metadata rows: 4
--------------------------------------------------------------------------------
File: card.csv
Columns: ['original_column_name', 'column_name', 'column_description', 'data_format', 'value_description']
Metadata rows: 4
--------------------------------------------------------------------------------
File: client.csv
Columns: ['original_column_name', 'column_name', 'column_description', 'data_format', 'value_description']
Metadata rows: 4
-----------------------------------------------------------------

## validate metadata against SQLite

In [19]:
import csv

expected_metadata_fields = {
    "original_column_name",
    "column_name",
    "column_description",
    "data_format",
    "value_description",
}

database_tables = {
    row[0]
    for row in conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        """
    )
}

metadata_tables = {
    path.stem
    for path in description_csvs
}

print("Database tables:", sorted(database_tables))
print("Metadata tables:", sorted(metadata_tables))
print("Missing metadata tables:", sorted(database_tables - metadata_tables))
print("Unexpected metadata tables:", sorted(metadata_tables - database_tables))

metadata_by_table = {}
validation_issues = []

for path in description_csvs:
    table_name = path.stem

    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

    sqlite_columns = [
        row[1]
        for row in conn.execute(
            f'PRAGMA table_info("{table_name}")'
        )
    ]

    metadata_columns = [
        (row.get("original_column_name") or "").strip()
        for row in rows
    ]

    blank_extra_field_values = [
        row.get("", "")
        for row in rows
        if (row.get("", "") or "").strip()
    ]

    if blank_extra_field_values:
        validation_issues.append({
            "table": table_name,
            "issue": "non-empty values under blank CSV header",
            "values": blank_extra_field_values,
        })

    if any(not column for column in metadata_columns):
        validation_issues.append({
            "table": table_name,
            "issue": "blank original_column_name",
        })

    duplicate_columns = sorted({
        column
        for column in metadata_columns
        if metadata_columns.count(column) > 1
    })

    if duplicate_columns:
        validation_issues.append({
            "table": table_name,
            "issue": "duplicate metadata columns",
            "columns": duplicate_columns,
        })

    sqlite_lookup = {
        column.casefold(): column
        for column in sqlite_columns
    }
    metadata_lookup = {
        column.casefold(): column
        for column in metadata_columns
    }

    missing_columns = [
        column
        for column in sqlite_columns
        if column.casefold() not in metadata_lookup
    ]

    unexpected_columns = [
        column
        for column in metadata_columns
        if column.casefold() not in sqlite_lookup
    ]

    if missing_columns:
        validation_issues.append({
            "table": table_name,
            "issue": "SQLite columns missing from metadata",
            "columns": missing_columns,
        })

    if unexpected_columns:
        validation_issues.append({
            "table": table_name,
            "issue": "metadata columns absent from SQLite",
            "columns": unexpected_columns,
        })

    metadata_by_table[table_name] = rows

    print("-" * 80)
    print("Table:", table_name)
    print("SQLite columns:", sqlite_columns)
    print("Metadata columns:", metadata_columns)
    print(
        "Case-insensitive column match:",
        not missing_columns and not unexpected_columns,
    )

print("=" * 80)
print(
    "Metadata/schema validation passed:",
    (
        database_tables == metadata_tables
        and not validation_issues
    ),
)
print("Validation issues:", validation_issues)

Database tables: ['account', 'card', 'client', 'disp', 'district', 'loan', 'order', 'trans']
Metadata tables: ['account', 'card', 'client', 'disp', 'district', 'loan', 'order', 'trans']
Missing metadata tables: []
Unexpected metadata tables: []
--------------------------------------------------------------------------------
Table: account
SQLite columns: ['account_id', 'district_id', 'frequency', 'date']
Metadata columns: ['account_id', 'district_id', 'frequency', 'date']
Case-insensitive column match: True
--------------------------------------------------------------------------------
Table: card
SQLite columns: ['card_id', 'disp_id', 'type', 'issued']
Metadata columns: ['card_id', 'disp_id', 'type', 'issued']
Case-insensitive column match: True
--------------------------------------------------------------------------------
Table: client
SQLite columns: ['client_id', 'gender', 'birth_date', 'district_id']
Metadata columns: ['client_id', 'gender', 'birth_date', 'district_id']
Case-in

In [20]:
for table_name in sorted(metadata_by_table):
    print("=" * 80)
    print("Table:", table_name)

    for row in metadata_by_table[table_name]:
        original_name = (
            row.get("original_column_name") or ""
        ).strip()
        friendly_name = (
            row.get("column_name") or ""
        ).strip()
        description = (
            row.get("column_description") or ""
        ).strip()

        print(
            f"{original_name} | "
            f"name={friendly_name!r} | "
            f"description={description!r}"
        )

Table: account
account_id | name='account id' | description='the id of the account'
district_id | name='location of branch' | description='location of branch'
frequency | name='frequency' | description='frequency of the acount'
date | name='date' | description='the creation date of the account'
Table: card
card_id | name='credit card id' | description='id number of credit card'
disp_id | name='disposition id' | description='disposition id'
type | name='' | description='type of credit card'
issued | name='' | description='the date when the credit card issued'
Table: client
client_id | name='' | description='the unique number'
gender | name='' | description=''
birth_date | name='' | description='birth date'
district_id | name='location of branch' | description='location of branch'
Table: disp
disp_id | name='disposition id' | description='unique number of identifying this row of record'
client_id | name='' | description='id number of client'
account_id | name='' | description='id number 

## Build the descriptions-only prompt

In [26]:
import hashlib
import re


def clean_metadata_text(value):
    return " ".join((value or "").strip().split())


def normalized_meaning(value):
    value = clean_metadata_text(value).casefold()
    return re.sub(r"[^a-z0-9]+", " ", value).strip()


def build_column_meanings(metadata_by_table):
    lines = ["Column meanings:"]

    for table_name in sorted(metadata_by_table):
        for row in metadata_by_table[table_name]:
            original_name = clean_metadata_text(
                row.get("original_column_name")
            )
            friendly_name = clean_metadata_text(
                row.get("column_name")
            )
            description = clean_metadata_text(
                row.get("column_description")
            )

            original_normalized = normalized_meaning(
                original_name.replace("_", " ")
            )

            meanings = []
            seen_meanings = set()

            for candidate in (friendly_name, description):
                candidate_normalized = normalized_meaning(candidate)

                if not candidate_normalized:
                    continue

                # The schema already exposes the original column name.
                if candidate_normalized == original_normalized:
                    continue

                # Avoid repeating identical friendly names/descriptions.
                if candidate_normalized in seen_meanings:
                    continue

                meanings.append(candidate)
                seen_meanings.add(candidate_normalized)

            if not meanings:
                continue

            quoted_table = table_name.replace('"', '""')
            quoted_column = original_name.replace('"', '""')

            lines.append(
                f'- "{quoted_table}"."{quoted_column}": '
                + "; ".join(meanings)
            )

    return "\n".join(lines)


column_meanings = build_column_meanings(metadata_by_table)

relationship_description_schema = (
    relationship_schema_context
    + "\n\n"
    + column_meanings
)

relationship_description_schema_hash = hashlib.sha256(
    relationship_description_schema.encode("utf-8")
).hexdigest()

print(relationship_description_schema)
print()
print(
    "Relationship + description schema SHA-256:",
    relationship_description_schema_hash,
)
print(
    "Schema characters:",
    len(relationship_description_schema),
)

SQLite schema:

CREATE TABLE "account" (
  "account_id" INTEGER PRIMARY KEY,
  "district_id" INTEGER,
  "frequency" TEXT,
  "date" DATE,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "card" (
  "card_id" INTEGER PRIMARY KEY,
  "disp_id" INTEGER,
  "type" TEXT,
  "issued" DATE,
  FOREIGN KEY ("disp_id") REFERENCES "disp" ("disp_id")
);

CREATE TABLE "client" (
  "client_id" INTEGER PRIMARY KEY,
  "gender" TEXT,
  "birth_date" DATE,
  "district_id" INTEGER,
  FOREIGN KEY ("district_id") REFERENCES "district" ("district_id")
);

CREATE TABLE "disp" (
  "disp_id" INTEGER PRIMARY KEY,
  "client_id" INTEGER,
  "account_id" INTEGER,
  "type" TEXT,
  FOREIGN KEY ("account_id") REFERENCES "account" ("account_id"),
  FOREIGN KEY ("client_id") REFERENCES "client" ("client_id")
);

CREATE TABLE "district" (
  "district_id" INTEGER PRIMARY KEY,
  "A2" TEXT,
  "A3" TEXT,
  "A4" TEXT,
  "A5" TEXT,
  "A6" TEXT,
  "A7" TEXT,
  "A8" INTEGER,
  "A9" INTEGER,
  "A10"

In [29]:
excluded_metadata_fields = {
    "data_format",
    "value_description",
    "",
}

forbidden_value_fragments = {
    "POPLATEK MESICNE",
    "POPLATEK TYDNE",
    "POPLATEK PO OBRATU",
    "DISPONENT",
}

assert all(
    fragment not in relationship_description_schema
    for fragment in forbidden_value_fragments
)

print("Descriptions-only leakage guard passed.")
print("Excluded metadata fields:", sorted(
    repr(field)
    for field in excluded_metadata_fields
))
print("No model request made.")

Descriptions-only leakage guard passed.
Excluded metadata fields: ["''", "'data_format'", "'value_description'"]
No model request made.


In [32]:
from domain_adapt.frontier import load_baseline_config
from domain_adapt.slm import request_sql as request_slm_sql


EXPECTED_DESCRIPTION_SCHEMA_SHA256 = (
    "86108a8314b1b09dcf48785fb8aac385c4bb0450a7d473b2f9bc1cf0c3f1a6a6"
)

actual_description_schema_sha256 = hashlib.sha256(
    relationship_description_schema.encode("utf-8")
).hexdigest()

assert (
    actual_description_schema_sha256
    == EXPECTED_DESCRIPTION_SCHEMA_SHA256
)

slm_config_path = Path(
    "../configs/slm_baseline_v1.json"
).resolve()

slm_config = load_baseline_config(slm_config_path)

assert slm_config["generation"]["attempts_per_question"] == 1
assert slm_config["generation"]["candidate_sql_repair"] is False

diagnostic_question_id = 14

working_ids = set(
    manifest["selection"]["working_dev"]["ids"]
)
frozen_ids = set(
    manifest["selection"]["frozen_eval"]["ids"]
)

assert diagnostic_question_id in working_ids
assert diagnostic_question_id not in frozen_ids

matches = [
    row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
    and row["question_id"] == diagnostic_question_id
]

assert len(matches) == 1
diagnostic_example = matches[0]

print("Question ID:", diagnostic_example["question_id"])
print("Difficulty:", diagnostic_example["difficulty"])
print("Question:", diagnostic_example["question"])
print("Schema SHA-256:", actual_description_schema_sha256)

Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Schema SHA-256: 86108a8314b1b09dcf48785fb8aac385c4bb0450a7d473b2f9bc1cf0c3f1a6a6


In [33]:
diagnostic_response, diagnostic_sql = request_slm_sql(
    client=client,
    deployment_name=slm_deployment_name,
    instructions=slm_config["instructions"],
    schema_context=relationship_description_schema,
    question=diagnostic_example["question"],
    max_tokens=slm_config["max_tokens"],
)

print("Response ID:", diagnostic_response.id)
print("Response model:", diagnostic_response.model)
print(
    "Finish reason:",
    diagnostic_response.choices[0].finish_reason,
)
print("\nRaw model output:")
print(diagnostic_sql)
print("\nUsage:")
print(diagnostic_response.usage)

Response ID: c9b6e5079962417ca82f451b9c8a8ce0
Response model: ministral-3b-2410
Finish reason: stop

Raw model output:
SELECT "account"."account_id", "account"."district_id", "account"."frequency", "account"."date"
FROM "account"
JOIN "trans" ON "account"."account_id" = "trans"."account_id"
WHERE "account"."date" < '1997-01-01'
AND "trans"."amount" > 3000

Usage:
CompletionUsage(completion_tokens=86, prompt_tokens=1588, total_tokens=1674, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [34]:
compatible_reference_sql, applied_rules = (
    make_reference_compatible(diagnostic_example["SQL"])
)

diagnostic_comparison = compare_sql_results(
    conn=conn,
    reference_sql=compatible_reference_sql,
    candidate_sql=diagnostic_sql,
)

print("Question ID:", diagnostic_question_id)
print("Reference-only compatibility rules:", applied_rules)
print("Passed:", diagnostic_comparison.equivalent)
print("Reason:", diagnostic_comparison.reason)
print(
    "Expected row count:",
    diagnostic_comparison.expected_row_count,
)
print(
    "Candidate row count:",
    diagnostic_comparison.candidate_row_count,
)

Question ID: 14
Reference-only compatibility rules: ['sqlite_four_digit_year']
Passed: False
Reason: column count differs: 1 != 4
Expected row count: 3591
Candidate row count: 419502


In [35]:
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
import json


description_run_path = Path(
    "../data/runs/"
    "ministral_3b_relationship_descriptions_working_dev_v1.json"
).resolve()

base_config_sha256 = hashlib.sha256(
    slm_config_path.read_bytes()
).hexdigest()

diagnostic_usage = (
    diagnostic_response.usage.model_dump()
    if diagnostic_response.usage is not None
    else {}
)

diagnostic_result = {
    "question_id": diagnostic_example["question_id"],
    "difficulty": diagnostic_example["difficulty"],
    "question": diagnostic_example["question"],
    "generated_sql": diagnostic_sql,
    "passed": diagnostic_comparison.equivalent,
    "reason": diagnostic_comparison.reason,
    "expected_row_count": (
        diagnostic_comparison.expected_row_count
    ),
    "candidate_row_count": (
        diagnostic_comparison.candidate_row_count
    ),
    "reference_compatibility_rules": applied_rules,
    "response_id": diagnostic_response.id,
    "response_model": diagnostic_response.model,
    "finish_reason": (
        diagnostic_response.choices[0].finish_reason
    ),
    "latency_seconds": None,
    "usage": diagnostic_usage,
    "api_error": None,
}


description_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "relationship_plus_column_descriptions",
    "split": "working_dev",
    "deployment_name": slm_deployment_name,
    "base_config_sha256": base_config_sha256,
    "relationship_schema_sha256": (
        "26cdd17a25945a10041bda7e9bf23e83"
        "aaacb0c3f261a899b04f2d78ac2a7d89"
    ),
    "metadata_sha256": (
        "3f7be83f3f9bdba77d27a66ad38b1a81"
        "b3a40361168780f54ceedd985db5fc23"
    ),
    "schema_sha256": (
        EXPECTED_DESCRIPTION_SCHEMA_SHA256
    ),
    "included_metadata_fields": [
        "column_name",
        "column_description",
    ],
    "excluded_metadata_fields": [
        "data_format",
        "value_description",
        "",
    ],
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "results": [diagnostic_result],
}


def save_description_run():
    description_run_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = description_run_path.with_suffix(
        ".json.tmp"
    )

    temporary_path.write_text(
        json.dumps(description_run, indent=2),
        encoding="utf-8",
    )

    temporary_path.replace(description_run_path)


save_description_run()

print("Recorded diagnostic question:", diagnostic_question_id)
print("Pending model requests: 11")
print("Saved:", description_run_path)

Recorded diagnostic question: 14
Pending model requests: 11
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_relationship_descriptions_working_dev_v1.json


In [36]:
working_examples = sorted(
    [
        row
        for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in working_ids
    ],
    key=lambda row: row["question_id"],
)

assert len(working_examples) == 12

completed_ids = {
    result["question_id"]
    for result in description_run["results"]
}

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

for index, working_example in enumerate(
    pending_examples,
    start=1,
):
    started = perf_counter()

    try:
        response, generated_sql = request_slm_sql(
            client=client,
            deployment_name=slm_deployment_name,
            instructions=slm_config["instructions"],
            schema_context=relationship_description_schema,
            question=working_example["question"],
            max_tokens=slm_config["max_tokens"],
        )

        latency_seconds = perf_counter() - started

        compatible_reference_sql, rules = (
            make_reference_compatible(
                working_example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": rules,
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": {},
            "api_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    description_run["results"].append(result)
    save_description_run()

    outcome = "PASS" if result["passed"] else "FAIL"

    print(
        f"[{index:02d}/{len(pending_examples):02d}] "
        f"ID {result['question_id']:>3} "
        f"({result['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

[01/11] ID   5 (medium) FAIL: unordered results differ
[02/11] ID  11 (medium) FAIL: ordered results differ
[03/11] ID  16 (easy  ) FAIL: candidate execution failed: no such column: account.date
[04/11] ID  45 (easy  ) FAIL: column count differs: 2 != 3
[05/11] ID  49 (medium) FAIL: unordered results differ
[06/11] ID  61 (hard  ) FAIL: candidate execution failed: no such column: a.type
[07/11] ID  69 (easy  ) PASS: unordered results match
[08/11] ID  81 (hard  ) FAIL: candidate execution failed: no such column: client.account_id
[09/11] ID  89 (easy  ) FAIL: unordered results differ
[10/11] ID  99 (easy  ) FAIL: unordered results differ
[11/11] ID 105 (medium) FAIL: candidate execution failed: no such column: d.district_id


In [37]:
from collections import Counter


description_results = sorted(
    description_run["results"],
    key=lambda result: result["question_id"],
)

assert len(description_results) == 12
assert {
    result["question_id"]
    for result in description_results
} == working_ids

passed_count = sum(
    result["passed"]
    for result in description_results
)

executable_count = sum(
    result["api_error"] is None
    and not result["reason"].startswith(
        "candidate execution failed"
    )
    for result in description_results
)

difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in description_results
        if result["difficulty"] == difficulty
    ]

    difficulty_summary[difficulty] = {
        "passed": sum(
            result["passed"]
            for result in subset
        ),
        "total": len(subset),
    }

failure_reasons = Counter(
    result["reason"]
    for result in description_results
    if not result["passed"]
)

total_usage = {
    token_name: sum(
        result["usage"].get(token_name, 0)
        for result in description_results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

print(
    f"Accuracy: {passed_count}/12 "
    f"({passed_count / 12:.1%})"
)
print(
    f"Executable: {executable_count}/12 "
    f"({executable_count / 12:.1%})"
)
print("By difficulty:", difficulty_summary)
print("Failure reasons:", dict(failure_reasons))
print(
    "API errors:",
    sum(
        result["api_error"] is not None
        for result in description_results
    ),
)
print("Token usage:", total_usage)
print("Saved:", description_run_path)

Accuracy: 1/12 (8.3%)
Executable: 8/12 (66.7%)
By difficulty: {'easy': {'passed': 1, 'total': 6}, 'medium': {'passed': 0, 'total': 4}, 'hard': {'passed': 0, 'total': 2}}
Failure reasons: {'unordered results differ': 4, 'ordered results differ': 1, 'column count differs: 1 != 4': 1, 'candidate execution failed: no such column: account.date': 1, 'column count differs: 2 != 3': 1, 'candidate execution failed: no such column: a.type': 1, 'candidate execution failed: no such column: client.account_id': 1, 'candidate execution failed: no such column: d.district_id': 1}
API errors: 0
Token usage: {'prompt_tokens': 19034, 'completion_tokens': 722, 'total_tokens': 19756}
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_relationship_descriptions_working_dev_v1.json


## inspect failures and value metadata

In [38]:
failures = [
    result
    for result in description_results
    if not result["passed"]
]

print("Failure count:", len(failures))

for failure in failures:
    print("=" * 80)
    print("Question ID:", failure["question_id"])
    print("Difficulty:", failure["difficulty"])
    print("Question:", failure["question"])
    print("Generated SQL:")
    print(failure["generated_sql"])
    print("Failure reason:", failure["reason"])
    print("API error:", failure["api_error"])

Failure count: 11
Question ID: 5
Difficulty: medium
Question: How many male customers who are living in North Bohemia have average salary greater than 8000?
Generated SQL:
SELECT COUNT(*)
FROM client
JOIN district ON client.district_id = district.district_id
WHERE client.gender = 'male'
AND district.A3 = 'North Bohemia'
AND district.A11 > 8000;
Failure reason: unordered results differ
API error: None
Question ID: 11
Difficulty: medium
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Generated SQL:
SELECT "account"."account_id"
FROM "account"
JOIN "loan" ON "account"."account_id" = "loan"."account_id"
WHERE "loan"."date" <= DATE('now', '-12 months')
AND "account"."date" = '1993-01-01'
ORDER BY "loan"."amount" DESC
LIMIT 1;
Failure reason: ordered results differ
API error: None
Question ID: 14
Difficulty: easy
Question: State different accounts who have account openi

In [39]:
excluded_metadata_records = []

for table_name in sorted(metadata_by_table):
    for row in metadata_by_table[table_name]:
        column_name = clean_metadata_text(
            row.get("original_column_name")
        )

        for field_name in (
            "data_format",
            "value_description",
            "",
        ):
            field_value = clean_metadata_text(
                row.get(field_name)
            )

            if not field_value:
                continue

            excluded_metadata_records.append({
                "table": table_name,
                "column": column_name,
                "field": (
                    field_name
                    if field_name
                    else "<blank overflow field>"
                ),
                "value": field_value,
            })

print(
    "Non-empty excluded metadata records:",
    len(excluded_metadata_records),
)

for record in excluded_metadata_records:
    print("=" * 80)
    print(
        f'{record["table"]}.{record["column"]} '
        f'[{record["field"]}]'
    )
    print(record["value"])

Non-empty excluded metadata records: 74
account.account_id [data_format]
integer
account.district_id [data_format]
integer
account.frequency [data_format]
text
account.frequency [<blank overflow field>]
"POPLATEK MESICNE" stands for monthly issuance "POPLATEK TYDNE" stands for weekly issuance "POPLATEK PO OBRATU" stands for issuance after transaction
account.date [data_format]
date
account.date [value_description]
in the form YYMMDD
card.card_id [data_format]
integer
card.disp_id [data_format]
integer
card.type [data_format]
text
card.type [value_description]
"junior": junior class of credit card; "classic": standard class of credit card; "gold": high-level credit card
card.issued [data_format]
date
card.issued [value_description]
in the form YYMMDD
client.client_id [data_format]
integer
client.gender [data_format]
text
client.gender [value_description]
F：female M：male
client.birth_date [data_format]
date
client.district_id [data_format]
integer
disp.disp_id [data_format]
integer
disp.

## profile actual categorical values

In [40]:
def quote_identifier(identifier):
    return '"' + identifier.replace('"', '""') + '"'


categorical_profile = {}

for table_name in sorted(database_tables):
    columns = conn.execute(
        f"PRAGMA table_info({quote_identifier(table_name)})"
    ).fetchall()

    text_columns = [
        row[1]
        for row in columns
        if (row[2] or "").upper() == "TEXT"
    ]

    for column_name in text_columns:
        quoted_table = quote_identifier(table_name)
        quoted_column = quote_identifier(column_name)

        distinct_count = conn.execute(
            f"""
            SELECT COUNT(DISTINCT {quoted_column})
            FROM {quoted_table}
            WHERE {quoted_column} IS NOT NULL
            """
        ).fetchone()[0]

        null_count = conn.execute(
            f"""
            SELECT COUNT(*)
            FROM {quoted_table}
            WHERE {quoted_column} IS NULL
            """
        ).fetchone()[0]

        profile = {
            "distinct_count": distinct_count,
            "null_count": null_count,
            "values": None,
        }

        # Inspect only genuinely categorical columns.
        if distinct_count <= 25:
            profile["values"] = conn.execute(
                f"""
                SELECT {quoted_column}, COUNT(*)
                FROM {quoted_table}
                WHERE {quoted_column} IS NOT NULL
                GROUP BY {quoted_column}
                ORDER BY COUNT(*) DESC, {quoted_column}
                """
            ).fetchall()

        categorical_profile[
            f"{table_name}.{column_name}"
        ] = profile


for qualified_name, profile in categorical_profile.items():
    print("=" * 80)
    print(qualified_name)
    print("Distinct count:", profile["distinct_count"])
    print("NULL count:", profile["null_count"])

    if profile["values"] is None:
        print("Values: not displayed — cardinality exceeds 25")
    else:
        print("Values:", profile["values"])

account.frequency
Distinct count: 3
NULL count: 0
Values: [('POPLATEK MESICNE', 4167), ('POPLATEK TYDNE', 240), ('POPLATEK PO OBRATU', 93)]
card.type
Distinct count: 3
NULL count: 0
Values: [('classic', 659), ('junior', 145), ('gold', 88)]
client.gender
Distinct count: 2
NULL count: 0
Values: [('M', 2724), ('F', 2645)]
disp.type
Distinct count: 2
NULL count: 0
Values: [('OWNER', 4500), ('DISPONENT', 869)]
district.A2
Distinct count: 77
NULL count: 0
Values: not displayed — cardinality exceeds 25
district.A3
Distinct count: 8
NULL count: 0
Values: [('south Moravia', 14), ('central Bohemia', 12), ('east Bohemia', 11), ('north Moravia', 11), ('north Bohemia', 10), ('west Bohemia', 10), ('south Bohemia', 8), ('Prague', 1)]
district.A4
Distinct count: 77
NULL count: 0
Values: not displayed — cardinality exceeds 25
district.A5
Distinct count: 53
NULL count: 0
Values: not displayed — cardinality exceeds 25
district.A6
Distinct count: 36
NULL count: 0
Values: not displayed — cardinality exceed

In [41]:
date_profile = {}

for table_name in sorted(database_tables):
    columns = conn.execute(
        f"PRAGMA table_info({quote_identifier(table_name)})"
    ).fetchall()

    date_columns = [
        row[1]
        for row in columns
        if (row[2] or "").upper() == "DATE"
    ]

    for column_name in date_columns:
        quoted_table = quote_identifier(table_name)
        quoted_column = quote_identifier(column_name)

        profile = conn.execute(
            f"""
            SELECT
                MIN({quoted_column}),
                MAX({quoted_column}),
                MIN(LENGTH({quoted_column})),
                MAX(LENGTH({quoted_column})),
                GROUP_CONCAT(
                    DISTINCT TYPEOF({quoted_column})
                )
            FROM {quoted_table}
            WHERE {quoted_column} IS NOT NULL
            """
        ).fetchone()

        date_profile[
            f"{table_name}.{column_name}"
        ] = {
            "minimum": profile[0],
            "maximum": profile[1],
            "minimum_length": profile[2],
            "maximum_length": profile[3],
            "storage_types": profile[4],
        }


for qualified_name, profile in date_profile.items():
    print("=" * 80)
    print(qualified_name)
    print(profile)

account.date
{'minimum': '1993-01-01', 'maximum': '1997-12-29', 'minimum_length': 10, 'maximum_length': 10, 'storage_types': 'text'}
card.issued
{'minimum': '1993-11-07', 'maximum': '1998-12-29', 'minimum_length': 10, 'maximum_length': 10, 'storage_types': 'text'}
client.birth_date
{'minimum': '1911-08-20', 'maximum': '1987-09-27', 'minimum_length': 10, 'maximum_length': 10, 'storage_types': 'text'}
loan.date
{'minimum': '1993-07-05', 'maximum': '1998-12-08', 'minimum_length': 10, 'maximum_length': 10, 'storage_types': 'text'}
trans.date
{'minimum': '1993-01-01', 'maximum': '1998-12-31', 'minimum_length': 10, 'maximum_length': 10, 'storage_types': 'text'}


In [42]:
verified_value_glossary = """Verified database values and formats:
- Use the exact stored text literals and casing shown below.
- All DATE columns are stored as TEXT in YYYY-MM-DD format.
- Monetary amount and balance columns use US dollars.
- "loan"."duration" is measured in months.
- "account"."frequency": 'POPLATEK MESICNE' = monthly issuance; 'POPLATEK TYDNE' = weekly issuance; 'POPLATEK PO OBRATU' = issuance after a transaction.
- "card"."type" uses 'junior', 'classic', or 'gold'.
- "client"."gender": 'M' = male; 'F' = female.
- "disp"."type": 'OWNER' = account owner; 'DISPONENT' = authorized user who is not the owner.
- "district"."A3" uses 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', or 'Prague'.
- "loan"."status": 'A' = finished with no problems; 'B' = finished but unpaid; 'C' = running and OK; 'D' = running with the client in debt.
- "order"."k_symbol": 'POJISTNE' = insurance; 'SIPO' = household payment; 'LEASING' = leasing; 'UVER' = loan payment.
- "trans"."type" uses 'PRIJEM', 'VYDAJ', or 'VYBER'; 'PRIJEM' means credit and 'VYDAJ' means withdrawal.
- "trans"."operation": 'VYBER KARTOU' = card withdrawal; 'VKLAD' = cash credit; 'PREVOD Z UCTU' = collection from another bank; 'VYBER' = cash withdrawal; 'PREVOD NA UCET' = remittance to another bank.
- "trans"."k_symbol": 'POJISTNE' = insurance; 'SLUZBY' = statement payment; 'UROK' = credited interest; 'SANKC. UROK' = sanction interest; 'SIPO' = household payment; 'DUCHOD' = pension; 'UVER' = loan payment.
"""

In [43]:
relationship_description_value_schema = (
    relationship_description_schema.rstrip()
    + "\n\n"
    + verified_value_glossary.strip()
)

relationship_description_value_schema_hash = hashlib.sha256(
    relationship_description_value_schema.encode("utf-8")
).hexdigest()

In [44]:
assert relationship_description_value_schema.count(
    "CREATE TABLE"
) == 8

assert "Column meanings:" in (
    relationship_description_value_schema
)

assert "Verified database values and formats:" in (
    relationship_description_value_schema
)

# Reject known stale or incorrect metadata.
for forbidden_fragment in (
    "YYMMDD",
    "account_to",
    "not useful",
    '"payments" is measured in months',
    "can only have the right",
):
    assert forbidden_fragment not in (
        verified_value_glossary
    )

# Confirm the important verified literals are present.
for required_fragment in (
    "'M' = male",
    "'POPLATEK TYDNE' = weekly issuance",
    "'C' = running and OK",
    "'D' = running with the client in debt",
    "'OWNER' = account owner",
    "'north Bohemia'",
    "YYYY-MM-DD",
):
    assert required_fragment in (
        verified_value_glossary
    )

print(verified_value_glossary)
print()
print(
    "Relationship + descriptions + values SHA-256:",
    relationship_description_value_schema_hash,
)
print(
    "Schema characters:",
    len(relationship_description_value_schema),
)
print("Verified-value safeguards passed.")
print("No model request made.")

Verified database values and formats:
- Use the exact stored text literals and casing shown below.
- All DATE columns are stored as TEXT in YYYY-MM-DD format.
- Monetary amount and balance columns use US dollars.
- "loan"."duration" is measured in months.
- "account"."frequency": 'POPLATEK MESICNE' = monthly issuance; 'POPLATEK TYDNE' = weekly issuance; 'POPLATEK PO OBRATU' = issuance after a transaction.
- "card"."type" uses 'junior', 'classic', or 'gold'.
- "client"."gender": 'M' = male; 'F' = female.
- "disp"."type": 'OWNER' = account owner; 'DISPONENT' = authorized user who is not the owner.
- "district"."A3" uses 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', or 'Prague'.
- "loan"."status": 'A' = finished with no problems; 'B' = finished but unpaid; 'C' = running and OK; 'D' = running with the client in debt.
- "order"."k_symbol": 'POJISTNE' = insurance; 'SIPO' = household payment; 'LEASING' = leasing; 'UVER' 

In [45]:
EXPECTED_VALUE_SCHEMA_SHA256 = (
    "ed139b776c279d00c25ce1a9247b41c13e294a8a6bb4a3f45053455286a790ae"
)

actual_value_schema_sha256 = hashlib.sha256(
    relationship_description_value_schema.encode("utf-8")
).hexdigest()

assert (
    actual_value_schema_sha256
    == EXPECTED_VALUE_SCHEMA_SHA256
)

value_diagnostic_question_id = 5

assert value_diagnostic_question_id in working_ids
assert value_diagnostic_question_id not in frozen_ids

matches = [
    row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
    and row["question_id"] == value_diagnostic_question_id
]

assert len(matches) == 1
value_diagnostic_example = matches[0]

print("Question ID:", value_diagnostic_example["question_id"])
print("Difficulty:", value_diagnostic_example["difficulty"])
print("Question:", value_diagnostic_example["question"])
print("Schema SHA-256:", actual_value_schema_sha256)

Question ID: 5
Difficulty: medium
Question: How many male customers who are living in North Bohemia have average salary greater than 8000?
Schema SHA-256: ed139b776c279d00c25ce1a9247b41c13e294a8a6bb4a3f45053455286a790ae


In [46]:
value_diagnostic_response, value_diagnostic_sql = (
    request_slm_sql(
        client=client,
        deployment_name=slm_deployment_name,
        instructions=slm_config["instructions"],
        schema_context=(
            relationship_description_value_schema
        ),
        question=value_diagnostic_example["question"],
        max_tokens=slm_config["max_tokens"],
    )
)

print("Response ID:", value_diagnostic_response.id)
print("Response model:", value_diagnostic_response.model)
print(
    "Finish reason:",
    value_diagnostic_response.choices[0].finish_reason,
)
print("\nRaw model output:")
print(value_diagnostic_sql)
print("\nUsage:")
print(value_diagnostic_response.usage)

Response ID: a3f1c810f70f446599102ea3dd38db82
Response model: ministral-3b-2410
Finish reason: stop

Raw model output:
SELECT COUNT(client_id)
FROM client
JOIN district ON client.district_id = district.district_id
WHERE client.gender = 'M'
AND district.A3 = 'north Bohemia'
AND district.A11 > 8000;

Usage:
CompletionUsage(completion_tokens=53, prompt_tokens=2084, total_tokens=2137, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [47]:
value_reference_sql, value_applied_rules = (
    make_reference_compatible(
        value_diagnostic_example["SQL"]
    )
)

value_diagnostic_comparison = compare_sql_results(
    conn=conn,
    reference_sql=value_reference_sql,
    candidate_sql=value_diagnostic_sql,
)

print(
    "Question ID:",
    value_diagnostic_question_id,
)
print(
    "Reference-only compatibility rules:",
    value_applied_rules,
)
print(
    "Passed:",
    value_diagnostic_comparison.equivalent,
)
print(
    "Reason:",
    value_diagnostic_comparison.reason,
)
print(
    "Expected row count:",
    value_diagnostic_comparison.expected_row_count,
)
print(
    "Candidate row count:",
    value_diagnostic_comparison.candidate_row_count,
)

Question ID: 5
Reference-only compatibility rules: ['case_insensitive_text_equality']
Passed: True
Reason: unordered results match
Expected row count: 1
Candidate row count: 1


## Initializing the value-aware run 

This is the first clean accuracy recovery attributable to a controlled harness layer:
- Same question and base instructions.
- Same one-attempt policy.
- SQL structure remained correct.
- Verified glossary changed male → M.
- It changed North Bohemia → north Bohemia.
- Result changed from FAIL to PASS.

The cost was 2,084 prompt tokens, so we now need the 12-question working result to judge quality versus cost.

In [48]:
value_run_path = Path(
    "../data/runs/"
    "ministral_3b_relationship_descriptions_values_working_dev_v1.json"
).resolve()

value_diagnostic_usage = (
    value_diagnostic_response.usage.model_dump()
    if value_diagnostic_response.usage is not None
    else {}
)

value_diagnostic_result = {
    "question_id": value_diagnostic_example["question_id"],
    "difficulty": value_diagnostic_example["difficulty"],
    "question": value_diagnostic_example["question"],
    "generated_sql": value_diagnostic_sql,
    "passed": value_diagnostic_comparison.equivalent,
    "reason": value_diagnostic_comparison.reason,
    "expected_row_count": (
        value_diagnostic_comparison.expected_row_count
    ),
    "candidate_row_count": (
        value_diagnostic_comparison.candidate_row_count
    ),
    "reference_compatibility_rules": value_applied_rules,
    "response_id": value_diagnostic_response.id,
    "response_model": value_diagnostic_response.model,
    "finish_reason": (
        value_diagnostic_response.choices[0].finish_reason
    ),
    "latency_seconds": None,
    "usage": value_diagnostic_usage,
    "api_error": None,
}

value_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": (
        "relationship_plus_descriptions_plus_verified_values"
    ),
    "split": "working_dev",
    "deployment_name": slm_deployment_name,
    "base_config_sha256": base_config_sha256,
    "metadata_sha256": (
        "3f7be83f3f9bdba77d27a66ad38b1a81"
        "b3a40361168780f54ceedd985db5fc23"
    ),
    "parent_schema_sha256": (
        "86108a8314b1b09dcf48785fb8aac385"
        "c4bb0450a7d473b2f9bc1cf0c3f1a6a6"
    ),
    "schema_sha256": EXPECTED_VALUE_SCHEMA_SHA256,
    "value_source": "database-verified metadata",
    "business_rules_included": False,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "results": [value_diagnostic_result],
}


def save_value_run():
    value_run_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = value_run_path.with_suffix(
        ".json.tmp"
    )

    temporary_path.write_text(
        json.dumps(value_run, indent=2),
        encoding="utf-8",
    )

    temporary_path.replace(value_run_path)


save_value_run()

print("Recorded diagnostic question:", value_diagnostic_question_id)
print("Pending model requests: 11")
print("Saved:", value_run_path)

Recorded diagnostic question: 5
Pending model requests: 11
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_relationship_descriptions_values_working_dev_v1.json


In [50]:
completed_ids = {
    result["question_id"]
    for result in value_run["results"]
}

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

assert len(pending_examples) == 11

for index, working_example in enumerate(
    pending_examples,
    start=1,
):
    started = perf_counter()

    try:
        response, generated_sql = request_slm_sql(
            client=client,
            deployment_name=slm_deployment_name,
            instructions=slm_config["instructions"],
            schema_context=(
                relationship_description_value_schema
            ),
            question=working_example["question"],
            max_tokens=slm_config["max_tokens"],
        )

        latency_seconds = perf_counter() - started

        compatible_reference_sql, rules = (
            make_reference_compatible(
                working_example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": rules,
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": {},
            "api_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    value_run["results"].append(result)
    save_value_run()

    outcome = "PASS" if result["passed"] else "FAIL"

    print(
        f"[{index:02d}/{len(pending_examples):02d}] "
        f"ID {result['question_id']:>3} "
        f"({result['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

[01/11] ID  11 (medium) FAIL: ordered results differ
[02/11] ID  14 (easy  ) FAIL: column count differs: 1 != 4
[03/11] ID  16 (easy  ) FAIL: candidate execution failed: no such column: account.date
[04/11] ID  45 (easy  ) FAIL: column count differs: 2 != 3
[05/11] ID  49 (medium) FAIL: candidate execution failed: no such column: loan_id
[06/11] ID  61 (hard  ) FAIL: unordered results differ
[07/11] ID  69 (easy  ) PASS: unordered results match
[08/11] ID  81 (hard  ) FAIL: candidate execution failed: no such column: client.account_id
[09/11] ID  89 (easy  ) FAIL: unordered results differ
[10/11] ID  99 (easy  ) FAIL: candidate execution failed: ambiguous column name: client_id
[11/11] ID 105 (medium) FAIL: candidate execution failed: near "order": syntax error


In [51]:
value_results = sorted(
    value_run["results"],
    key=lambda result: result["question_id"],
)

assert len(value_results) == 12
assert {
    result["question_id"]
    for result in value_results
} == working_ids

value_passed_count = sum(
    result["passed"]
    for result in value_results
)

value_executable_count = sum(
    result["api_error"] is None
    and not result["reason"].startswith(
        "candidate execution failed"
    )
    for result in value_results
)

value_difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in value_results
        if result["difficulty"] == difficulty
    ]

    value_difficulty_summary[difficulty] = {
        "passed": sum(
            result["passed"]
            for result in subset
        ),
        "total": len(subset),
    }

value_failure_reasons = Counter(
    result["reason"]
    for result in value_results
    if not result["passed"]
)

value_total_usage = {
    token_name: sum(
        result["usage"].get(token_name, 0)
        for result in value_results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

print(
    f"Accuracy: {value_passed_count}/12 "
    f"({value_passed_count / 12:.1%})"
)
print(
    f"Executable: {value_executable_count}/12 "
    f"({value_executable_count / 12:.1%})"
)
print("By difficulty:", value_difficulty_summary)
print(
    "Failure reasons:",
    dict(value_failure_reasons),
)
print(
    "API errors:",
    sum(
        result["api_error"] is not None
        for result in value_results
    ),
)
print("Token usage:", value_total_usage)
print("Saved:", value_run_path)

Accuracy: 2/12 (16.7%)
Executable: 7/12 (58.3%)
By difficulty: {'easy': {'passed': 1, 'total': 6}, 'medium': {'passed': 1, 'total': 4}, 'hard': {'passed': 0, 'total': 2}}
Failure reasons: {'ordered results differ': 1, 'column count differs: 1 != 4': 1, 'candidate execution failed: no such column: account.date': 1, 'column count differs: 2 != 3': 1, 'candidate execution failed: no such column: loan_id': 1, 'unordered results differ': 2, 'candidate execution failed: no such column: client.account_id': 1, 'candidate execution failed: ambiguous column name: client_id': 1, 'candidate execution failed: near "order": syntax error': 1}
API errors: 0
Token usage: {'prompt_tokens': 25070, 'completion_tokens': 903, 'total_tokens': 25973}
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_relationship_descriptions_values_working_dev_v1.json


In [52]:
value_failures = [
    result
    for result in value_results
    if not result["passed"]
]

print("Failure count:", len(value_failures))

for failure in value_failures:
    print("=" * 80)
    print("Question ID:", failure["question_id"])
    print("Difficulty:", failure["difficulty"])
    print("Question:", failure["question"])
    print("Generated SQL:")
    print(failure["generated_sql"])
    print("Failure reason:", failure["reason"])
    print("API error:", failure["api_error"])

Failure count: 10
Question ID: 11
Difficulty: medium
Question: Among the accounts who have loan validity more than 12 months, list out the accounts that have the highest approved amount and have account opening date in 1993.
Generated SQL:
SELECT "account"."account_id"
FROM "account"
JOIN "loan" ON "account"."account_id" = "loan"."account_id"
WHERE "loan"."date" > '1993-01-01'
AND "loan"."amount" = (
  SELECT MAX("loan"."amount")
  FROM "loan"
  WHERE "loan"."date" > '1993-01-01'
)
Failure reason: ordered results differ
API error: None
Question ID: 14
Difficulty: easy
Question: State different accounts who have account opening date before 1997 and own an amount of money greater than 3000USD
Generated SQL:
SELECT "account"."account_id", "account"."district_id", "account"."frequency", "account"."date"
FROM "account"
WHERE "account"."date" < '1997-01-01'
AND EXISTS (
  SELECT 1
  FROM "trans"
  WHERE "trans"."account_id" = "account"."account_id"
  AND "trans"."type" = 'PRIJEM'
  AND "tran

In [53]:
relationship_run_path = Path(
    "../data/runs/"
    "ministral_3b_relationship_schema_working_dev_v1.json"
).resolve()

relationship_run = json.loads(
    relationship_run_path.read_text(encoding="utf-8")
)

relationship_results_by_id = {
    result["question_id"]: result
    for result in relationship_run["results"]
}

description_results_by_id = {
    result["question_id"]: result
    for result in description_results
}

value_results_by_id = {
    result["question_id"]: result
    for result in value_results
}


def result_executable(result):
    return (
        result["api_error"] is None
        and not result["reason"].startswith(
            "candidate execution failed"
        )
    )


print(
    "ID | Rel pass/exec | Desc pass/exec | "
    "Value pass/exec"
)

for question_id in sorted(working_ids):
    relationship_result = (
        relationship_results_by_id[question_id]
    )
    description_result = (
        description_results_by_id[question_id]
    )
    value_result = value_results_by_id[question_id]

    print(
        f"{question_id:>3} | "
        f"{str(relationship_result['passed']):<5}/"
        f"{str(result_executable(relationship_result)):<5} | "
        f"{str(description_result['passed']):<5}/"
        f"{str(result_executable(description_result)):<5} | "
        f"{str(value_result['passed']):<5}/"
        f"{str(result_executable(value_result)):<5}"
    )

ID | Rel pass/exec | Desc pass/exec | Value pass/exec
  5 | False/True  | False/True  | True /True 
 11 | False/False | False/True  | False/True 
 14 | False/True  | False/True  | False/True 
 16 | True /True  | False/False | False/False
 45 | False/False | False/True  | False/True 
 49 | False/True  | False/True  | False/False
 61 | False/False | False/False | False/True 
 69 | False/True  | True /True  | True /True 
 81 | False/True  | False/False | False/False
 89 | False/True  | False/True  | False/True 
 99 | False/True  | False/True  | False/False
105 | False/False | False/False | False/False


In [54]:
compact_domain_context = """SQLite financial database.

Schema:
- account: account_id INTEGER PK; district_id INTEGER FK->district.district_id; frequency TEXT; date DATE.
- card: card_id INTEGER PK; disp_id INTEGER FK->disp.disp_id; type TEXT; issued DATE.
- client: client_id INTEGER PK; gender TEXT; birth_date DATE; district_id INTEGER FK->district.district_id.
- disp: disp_id INTEGER PK; client_id INTEGER FK->client.client_id; account_id INTEGER FK->account.account_id; type TEXT.
- district: district_id INTEGER PK; A2 TEXT; A3 TEXT; A4 TEXT; A5 TEXT; A6 TEXT; A7 TEXT; A8 INTEGER; A9 INTEGER; A10 REAL; A11 INTEGER; A12 REAL; A13 REAL; A14 INTEGER; A15 INTEGER; A16 INTEGER.
- loan: loan_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; amount INTEGER; duration INTEGER; payments REAL; status TEXT.
- "order": order_id INTEGER PK; account_id INTEGER FK->account.account_id; bank_to TEXT; account_to INTEGER; amount REAL; k_symbol TEXT.
- trans: trans_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; type TEXT; operation TEXT; amount INTEGER; balance INTEGER; k_symbol TEXT; bank TEXT; account INTEGER.

Meanings:
- account.frequency = statement issuance frequency; account.date = account opening date.
- card.type = credit-card class; card.issued = card issue date.
- client.gender = client gender; client.birth_date = client birth date.
- disp.type = client's role for an account.
- district.A2 = district name; A3 = region; A4 = inhabitants; A5 = municipalities with fewer than 499 inhabitants; A6 = municipalities with 500-1999; A7 = municipalities with 2000-9999; A8 = municipalities with more than 10000; A10 = urban-inhabitant ratio; A11 = average salary; A12 = unemployment rate in 1995; A13 = unemployment rate in 1996; A14 = entrepreneurs per 1000 inhabitants; A15 = committed crimes in 1995; A16 = committed crimes in 1996.
- loan.date = approval date; loan.amount = approved amount; loan.duration = duration in months; loan.payments = monthly payment amount; loan.status = repayment status.
- "order".bank_to = recipient bank; account_to = recipient account; amount = debited amount; k_symbol = payment purpose.
- trans.date = transaction date; trans.type = credit or withdrawal; trans.operation = transaction mode; trans.amount = transaction amount; trans.balance = balance after the transaction; trans.k_symbol = transaction purpose; trans.bank = partner bank; trans.account = partner account.

Verified values and formats:
- Use exact stored text literals and casing.
- Dates are TEXT in YYYY-MM-DD format. Monetary amounts and balances use US dollars.
- account.frequency: 'POPLATEK MESICNE'=monthly; 'POPLATEK TYDNE'=weekly; 'POPLATEK PO OBRATU'=after transaction.
- card.type: 'junior', 'classic', 'gold'.
- client.gender: 'M'=male; 'F'=female.
- disp.type: 'OWNER'=owner; 'DISPONENT'=authorized non-owner.
- district.A3: 'south Moravia', 'central Bohemia', 'east Bohemia', 'north Moravia', 'north Bohemia', 'west Bohemia', 'south Bohemia', 'Prague'.
- loan.status: 'A'=finished without problems; 'B'=finished unpaid; 'C'=running and OK; 'D'=running in debt.
- "order".k_symbol: 'POJISTNE'=insurance; 'SIPO'=household; 'LEASING'=leasing; 'UVER'=loan.
- trans.type: 'PRIJEM'=credit; 'VYDAJ'=withdrawal; 'VYBER' is also stored.
- trans.operation: 'VYBER KARTOU'=card withdrawal; 'VKLAD'=cash credit; 'PREVOD Z UCTU'=collection from another bank; 'VYBER'=cash withdrawal; 'PREVOD NA UCET'=remittance to another bank.
- trans.k_symbol: 'POJISTNE'=insurance; 'SLUZBY'=statement payment; 'UROK'=credited interest; 'SANKC. UROK'=sanction interest; 'SIPO'=household; 'DUCHOD'=pension; 'UVER'=loan.
"""

In [55]:
compact_domain_context_hash = hashlib.sha256(
    compact_domain_context.encode("utf-8")
).hexdigest()

required_schema_fragments = (
    "account_id INTEGER PK",
    "district_id INTEGER FK->district.district_id",
    "disp_id INTEGER FK->disp.disp_id",
    "client_id INTEGER FK->client.client_id",
    "account_id INTEGER FK->account.account_id",
    "A11 = average salary",
    "A13 = unemployment rate in 1996",
    "trans.balance = balance after the transaction",
    "'M'=male",
    "'POPLATEK TYDNE'=weekly",
    "'C'=running and OK",
    "'D'=running in debt",
)

for fragment in required_schema_fragments:
    assert fragment in compact_domain_context

for forbidden_fragment in (
    "YYMMDD",
    "can only have the right",
    "Reference SQL",
    "Question ID",
):
    assert forbidden_fragment not in compact_domain_context

assert len(compact_domain_context) < len(
    relationship_description_value_schema
)

reduction = 1 - (
    len(compact_domain_context)
    / len(relationship_description_value_schema)
)

print(compact_domain_context)
print()
print(
    "Compact domain-context SHA-256:",
    compact_domain_context_hash,
)
print(
    "Compact characters:",
    len(compact_domain_context),
)
print(
    "Previous characters:",
    len(relationship_description_value_schema),
)
print(f"Character reduction: {reduction:.1%}")
print("Compact-context safeguards passed.")
print("No model request made.")

SQLite financial database.

Schema:
- account: account_id INTEGER PK; district_id INTEGER FK->district.district_id; frequency TEXT; date DATE.
- card: card_id INTEGER PK; disp_id INTEGER FK->disp.disp_id; type TEXT; issued DATE.
- client: client_id INTEGER PK; gender TEXT; birth_date DATE; district_id INTEGER FK->district.district_id.
- disp: disp_id INTEGER PK; client_id INTEGER FK->client.client_id; account_id INTEGER FK->account.account_id; type TEXT.
- district: district_id INTEGER PK; A2 TEXT; A3 TEXT; A4 TEXT; A5 TEXT; A6 TEXT; A7 TEXT; A8 INTEGER; A9 INTEGER; A10 REAL; A11 INTEGER; A12 REAL; A13 REAL; A14 INTEGER; A15 INTEGER; A16 INTEGER.
- loan: loan_id INTEGER PK; account_id INTEGER FK->account.account_id; date DATE; amount INTEGER; duration INTEGER; payments REAL; status TEXT.
- "order": order_id INTEGER PK; account_id INTEGER FK->account.account_id; bank_to TEXT; account_to INTEGER; amount REAL; k_symbol TEXT.
- trans: trans_id INTEGER PK; account_id INTEGER FK->account.acc

In [62]:
EXPECTED_COMPACT_CONTEXT_SHA256 = (
    "634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987"
)

actual_compact_context_sha256 = hashlib.sha256(
    compact_domain_context.encode("utf-8")
).hexdigest()

assert (
    actual_compact_context_sha256
    == EXPECTED_COMPACT_CONTEXT_SHA256
)

compact_diagnostic_question_id = 16

assert compact_diagnostic_question_id in working_ids
assert compact_diagnostic_question_id not in frozen_ids

matches = [
    row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
    and row["question_id"] == compact_diagnostic_question_id
]

assert len(matches) == 1
compact_diagnostic_example = matches[0]

print("Question ID:", compact_diagnostic_example["question_id"])
print("Difficulty:", compact_diagnostic_example["difficulty"])
print("Question:", compact_diagnostic_example["question"])
print("Context SHA-256:", actual_compact_context_sha256)

Question ID: 16
Difficulty: easy
Question: The transaction of 840 USD happened in 1998/10/14, when was this account opened?
Context SHA-256: 634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987


In [63]:
compact_diagnostic_response, compact_diagnostic_sql = (
    request_slm_sql(
        client=client,
        deployment_name=slm_deployment_name,
        instructions=slm_config["instructions"],
        schema_context=compact_domain_context,
        question=compact_diagnostic_example["question"],
        max_tokens=slm_config["max_tokens"],
    )
)

print("Response ID:", compact_diagnostic_response.id)
print("Response model:", compact_diagnostic_response.model)
print(
    "Finish reason:",
    compact_diagnostic_response.choices[0].finish_reason,
)
print("\nRaw model output:")
print(compact_diagnostic_sql)
print("\nUsage:")
print(compact_diagnostic_response.usage)

Response ID: de523fc9025d4957bcb865d4911555dc
Response model: ministral-3b-2410
Finish reason: stop

Raw model output:
SELECT account.date
FROM trans
WHERE trans.date = '1998-10-14' AND trans.amount = 840;

Usage:
CompletionUsage(completion_tokens=33, prompt_tokens=1150, total_tokens=1183, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [64]:
compact_reference_sql, compact_applied_rules = (
    make_reference_compatible(
        compact_diagnostic_example["SQL"]
    )
)

compact_diagnostic_comparison = compare_sql_results(
    conn=conn,
    reference_sql=compact_reference_sql,
    candidate_sql=compact_diagnostic_sql,
)

print("Question ID:", compact_diagnostic_question_id)
print(
    "Reference-only compatibility rules:",
    compact_applied_rules,
)
print(
    "Passed:",
    compact_diagnostic_comparison.equivalent,
)
print(
    "Reason:",
    compact_diagnostic_comparison.reason,
)
print(
    "Expected row count:",
    compact_diagnostic_comparison.expected_row_count,
)
print(
    "Candidate row count:",
    compact_diagnostic_comparison.candidate_row_count,
)

Question ID: 16
Reference-only compatibility rules: ['case_insensitive_text_equality']
Passed: False
Reason: candidate execution failed: no such column: account.date
Expected row count: 1
Candidate row count: None


In [65]:
sql_construction_guardrails = """SQL construction checks:
- Every table referenced in SELECT, WHERE, JOIN, GROUP BY, HAVING, or ORDER BY must appear in FROM or JOIN.
- Join tables only through the foreign-key relationships supplied in the schema.
- Verify that every column belongs to the table or alias used to qualify it.
- When more than one table is used, qualify every selected, filtered, grouped, and ordered column.
- Return only the columns requested by the question, in the requested order. Do not return helper, filter, ranking, or aggregate columns unless requested.
- Do not add filters or assumptions that the question does not request.
- If the question asks for different, distinct, or unique items, use DISTINCT.
- For highest or lowest, sort the relevant expression and apply LIMIT only when one result is requested.
- For a calendar year stored in YYYY-MM-DD text, use a year extraction or a correct start-inclusive/end-exclusive date range.
- Quote identifiers whenever SQLite requires quoting.
- Before returning, check that the query is executable against the supplied schema."""

In [66]:
guardrailed_instructions = (
    slm_config["instructions"].rstrip()
    + "\n\n"
    + sql_construction_guardrails
)

guardrailed_instructions_hash = hashlib.sha256(
    guardrailed_instructions.encode("utf-8")
).hexdigest()

print(guardrailed_instructions)
print()
print(
    "Guardrailed-instructions SHA-256:",
    guardrailed_instructions_hash,
)
print(
    "Base instruction characters:",
    len(slm_config["instructions"]),
)
print(
    "Guardrailed instruction characters:",
    len(guardrailed_instructions),
)

You translate natural-language questions into SQLite SQL.

Return exactly one executable SQLite query.
Use only the tables and columns in the supplied schema.
Do not include Markdown, code fences, comments, or explanations.
Do not invent tables or columns.

SQL construction checks:
- Every table referenced in SELECT, WHERE, JOIN, GROUP BY, HAVING, or ORDER BY must appear in FROM or JOIN.
- Join tables only through the foreign-key relationships supplied in the schema.
- Verify that every column belongs to the table or alias used to qualify it.
- When more than one table is used, qualify every selected, filtered, grouped, and ordered column.
- Return only the columns requested by the question, in the requested order. Do not return helper, filter, ranking, or aggregate columns unless requested.
- Do not add filters or assumptions that the question does not request.
- If the question asks for different, distinct, or unique items, use DISTINCT.
- For highest or lowest, sort the relevant exp

In [67]:
# The rules must remain general rather than targeting a working question.
for forbidden_fragment in (
    "account.date",
    "trans.date",
    "client.gender",
    "loan.status",
    "POPLATEK",
    "North Bohemia",
    "Question ID",
    "reference SQL",
):
    assert forbidden_fragment.casefold() not in (
        guardrailed_instructions.casefold()
    )

required_guardrails = (
    "must appear in FROM or JOIN",
    "foreign-key relationships",
    "qualify every selected",
    "only the columns requested",
    "Do not add filters",
    "use DISTINCT",
    "YYYY-MM-DD",
    "query is executable",
)

for required_guardrail in required_guardrails:
    assert required_guardrail in guardrailed_instructions

print("General guardrail safeguards passed.")
print(
    "Compact context SHA-256:",
    compact_domain_context_hash,
)
print("No model request made.")

General guardrail safeguards passed.
Compact context SHA-256: 634503b9d904730790555dfeabb2a81a920293a9d747416a04e372e122d7b987
No model request made.


In [69]:
EXPECTED_GUARDRAIL_HASH = (
    "175e0489882789131bba788ad087f79e24a2207460d934778a16e04d7f265509"
)

assert hashlib.sha256(
    guardrailed_instructions.encode("utf-8")
).hexdigest() == EXPECTED_GUARDRAIL_HASH

assert hashlib.sha256(
    compact_domain_context.encode("utf-8")
).hexdigest() == EXPECTED_COMPACT_CONTEXT_SHA256

In [70]:
guardrail_response, guardrail_sql = request_slm_sql(
    client=client,
    deployment_name=slm_deployment_name,
    instructions=guardrailed_instructions,
    schema_context=compact_domain_context,
    question=compact_diagnostic_example["question"],
    max_tokens=slm_config["max_tokens"],
)

print("Response ID:", guardrail_response.id)
print("Response model:", guardrail_response.model)
print(
    "Finish reason:",
    guardrail_response.choices[0].finish_reason,
)
print("\nRaw model output:")
print(guardrail_sql)
print("\nUsage:")
print(guardrail_response.usage)

Response ID: df4565b5a6034d8c894d40042a60fc85
Response model: ministral-3b-2410
Finish reason: stop

Raw model output:
SELECT account.date
FROM account
JOIN trans ON account.account_id = trans.account_id
WHERE trans.amount = 840 AND trans.date = '1998-10-14';

Usage:
CompletionUsage(completion_tokens=44, prompt_tokens=1369, total_tokens=1413, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [71]:
guardrail_reference_sql, guardrail_applied_rules = (
    make_reference_compatible(
        compact_diagnostic_example["SQL"]
    )
)

guardrail_comparison = compare_sql_results(
    conn=conn,
    reference_sql=guardrail_reference_sql,
    candidate_sql=guardrail_sql,
)

print("Question ID:", compact_diagnostic_question_id)
print(
    "Reference-only compatibility rules:",
    guardrail_applied_rules,
)
print("Passed:", guardrail_comparison.equivalent)
print("Reason:", guardrail_comparison.reason)
print(
    "Expected row count:",
    guardrail_comparison.expected_row_count,
)
print(
    "Candidate row count:",
    guardrail_comparison.candidate_row_count,
)

Question ID: 16
Reference-only compatibility rules: ['case_insensitive_text_equality']
Passed: True
Reason: unordered results match
Expected row count: 1
Candidate row count: 1


In [74]:
guardrail_run_path = Path(
    "../data/runs/"
    "ministral_3b_compact_guardrails_working_dev_v1.json"
).resolve()

guardrail_usage = (
    guardrail_response.usage.model_dump()
    if guardrail_response.usage is not None
    else {}
)

guardrail_diagnostic_result = {
    "question_id": compact_diagnostic_example["question_id"],
    "difficulty": compact_diagnostic_example["difficulty"],
    "question": compact_diagnostic_example["question"],
    "generated_sql": guardrail_sql,
    "passed": guardrail_comparison.equivalent,
    "reason": guardrail_comparison.reason,
    "expected_row_count": guardrail_comparison.expected_row_count,
    "candidate_row_count": guardrail_comparison.candidate_row_count,
    "reference_compatibility_rules": guardrail_applied_rules,
    "response_id": guardrail_response.id,
    "response_model": guardrail_response.model,
    "finish_reason": (
        guardrail_response.choices[0].finish_reason
    ),
    "latency_seconds": None,
    "usage": guardrail_usage,
    "api_error": None,
}

guardrail_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "compact_domain_context_plus_sql_guardrails",
    "split": "working_dev",
    "deployment_name": slm_deployment_name,
    "base_config_sha256": base_config_sha256,
    "context_sha256": EXPECTED_COMPACT_CONTEXT_SHA256,
    "instructions_sha256": EXPECTED_GUARDRAIL_HASH,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "results": [guardrail_diagnostic_result],
}


def save_guardrail_run():
    guardrail_run_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = guardrail_run_path.with_suffix(
        ".json.tmp"
    )

    temporary_path.write_text(
        json.dumps(guardrail_run, indent=2),
        encoding="utf-8",
    )

    temporary_path.replace(guardrail_run_path)


save_guardrail_run()

print("Recorded diagnostic question:", compact_diagnostic_question_id)
print("Pending model requests: 11")
print("Saved:", guardrail_run_path)

Recorded diagnostic question: 16
Pending model requests: 11
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_compact_guardrails_working_dev_v1.json


In [75]:
completed_ids = {
    result["question_id"]
    for result in guardrail_run["results"]
}

pending_examples = [
    example
    for example in working_examples
    if example["question_id"] not in completed_ids
]

assert len(pending_examples) == 11

for index, working_example in enumerate(
    pending_examples,
    start=1,
):
    started = perf_counter()

    try:
        response, generated_sql = request_slm_sql(
            client=client,
            deployment_name=slm_deployment_name,
            instructions=guardrailed_instructions,
            schema_context=compact_domain_context,
            question=working_example["question"],
            max_tokens=slm_config["max_tokens"],
        )

        latency_seconds = perf_counter() - started

        compatible_reference_sql, rules = (
            make_reference_compatible(
                working_example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=compatible_reference_sql,
            candidate_sql=generated_sql,
        )

        usage = (
            response.usage.model_dump()
            if response.usage is not None
            else {}
        )

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": generated_sql,
            "passed": comparison.equivalent,
            "reason": comparison.reason,
            "expected_row_count": comparison.expected_row_count,
            "candidate_row_count": comparison.candidate_row_count,
            "reference_compatibility_rules": rules,
            "response_id": response.id,
            "response_model": response.model,
            "finish_reason": (
                response.choices[0].finish_reason
            ),
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": usage,
            "api_error": None,
        }

    except Exception as error:
        latency_seconds = perf_counter() - started

        result = {
            "question_id": working_example["question_id"],
            "difficulty": working_example["difficulty"],
            "question": working_example["question"],
            "generated_sql": None,
            "passed": False,
            "reason": "model request failed",
            "expected_row_count": None,
            "candidate_row_count": None,
            "reference_compatibility_rules": None,
            "response_id": None,
            "response_model": None,
            "finish_reason": None,
            "latency_seconds": round(
                latency_seconds,
                3,
            ),
            "usage": {},
            "api_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    guardrail_run["results"].append(result)
    save_guardrail_run()

    outcome = "PASS" if result["passed"] else "FAIL"

    print(
        f"[{index:02d}/{len(pending_examples):02d}] "
        f"ID {result['question_id']:>3} "
        f"({result['difficulty']:<6}) "
        f"{outcome}: {result['reason']}"
    )

[01/11] ID   5 (medium) PASS: unordered results match
[02/11] ID  11 (medium) FAIL: candidate execution failed: ambiguous column name: account_id
[03/11] ID  14 (easy  ) FAIL: candidate execution failed: near "```sql
SELECT DISTINCT account.account_id, account.date
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE account.date < '1997-01-01'
  AND loan.amount > 3000;
```": syntax error
[04/11] ID  45 (easy  ) FAIL: ordered results differ
[05/11] ID  49 (medium) FAIL: unordered results differ
[06/11] ID  61 (hard  ) FAIL: candidate execution failed: near "```sql
SELECT DISTINCT account.frequency
FROM account
JOIN district ON account.district_id = district.district_id
WHERE district.A11 BETWEEN 8000 AND 9000;
```": syntax error
[07/11] ID  69 (easy  ) PASS: unordered results match
[08/11] ID  81 (hard  ) FAIL: candidate execution failed: no such column: account.account_id
[09/11] ID  89 (easy  ) FAIL: candidate execution failed: near "```sql
SELECT SUM(trans.amount) AS

In [76]:
guardrail_results = sorted(
    guardrail_run["results"],
    key=lambda result: result["question_id"],
)

assert len(guardrail_results) == 12
assert {
    result["question_id"]
    for result in guardrail_results
} == working_ids

guardrail_passed_count = sum(
    result["passed"]
    for result in guardrail_results
)

guardrail_executable_count = sum(
    result["api_error"] is None
    and not result["reason"].startswith(
        "candidate execution failed"
    )
    for result in guardrail_results
)

guardrail_difficulty_summary = {}

for difficulty in ("easy", "medium", "hard"):
    subset = [
        result
        for result in guardrail_results
        if result["difficulty"] == difficulty
    ]

    guardrail_difficulty_summary[difficulty] = {
        "passed": sum(result["passed"] for result in subset),
        "total": len(subset),
    }

guardrail_failure_reasons = Counter(
    result["reason"]
    for result in guardrail_results
    if not result["passed"]
)

guardrail_total_usage = {
    token_name: sum(
        result["usage"].get(token_name, 0)
        for result in guardrail_results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

print(
    f"Accuracy: {guardrail_passed_count}/12 "
    f"({guardrail_passed_count / 12:.1%})"
)
print(
    f"Executable: {guardrail_executable_count}/12 "
    f"({guardrail_executable_count / 12:.1%})"
)
print(
    "By difficulty:",
    guardrail_difficulty_summary,
)
print(
    "Failure reasons:",
    dict(guardrail_failure_reasons),
)
print(
    "API errors:",
    sum(
        result["api_error"] is not None
        for result in guardrail_results
    ),
)
print("Token usage:", guardrail_total_usage)
print("Saved:", guardrail_run_path)

Accuracy: 4/12 (33.3%)
Executable: 6/12 (50.0%)
By difficulty: {'easy': {'passed': 3, 'total': 6}, 'medium': {'passed': 1, 'total': 4}, 'hard': {'passed': 0, 'total': 2}}
Failure reasons: {'candidate execution failed: ambiguous column name: account_id': 1, 'candidate execution failed: near "```sql\nSELECT DISTINCT account.account_id, account.date\nFROM account\nJOIN loan ON account.account_id = loan.account_id\nWHERE account.date < \'1997-01-01\'\n  AND loan.amount > 3000;\n```": syntax error': 1, 'ordered results differ': 1, 'unordered results differ': 1, 'candidate execution failed: near "```sql\nSELECT DISTINCT account.frequency\nFROM account\nJOIN district ON account.district_id = district.district_id\nWHERE district.A11 BETWEEN 8000 AND 9000;\n```": syntax error': 1, 'candidate execution failed: no such column: account.account_id': 1, 'candidate execution failed: near "```sql\nSELECT SUM(trans.amount) AS total_amount, client.gender\nFROM trans\nJOIN account ON trans.account_id = a

## Strictly unwrap one fenced SQL block

Four of the six execution failures are caused solely by Markdown envelopes. The model ignored the explicit no-fence instruction. Before adding more prompting, test a strict output-envelope normalizer offline. This makes no model requests and performs no semantic SQL repair.

In [77]:
def unwrap_single_sql_fence(text):
    """Remove one surrounding SQL fence and nothing else."""
    stripped = (text or "").strip()
    lines = stripped.splitlines()

    if len(lines) < 3:
        return stripped, False

    opening = lines[0].strip().casefold()
    closing = lines[-1].strip()

    if opening not in {
        "```",
        "```sql",
        "```sqlite",
    }:
        return stripped, False

    if closing != "```":
        return stripped, False

    inner_lines = lines[1:-1]

    # Refuse nested or multiple fenced blocks.
    if any("```" in line for line in inner_lines):
        return stripped, False

    inner_sql = "\n".join(inner_lines).strip()

    if not inner_sql:
        return stripped, False

    return inner_sql, True

In [78]:
assert unwrap_single_sql_fence(
    "```sql\nSELECT 1;\n```"
) == ("SELECT 1;", True)

assert unwrap_single_sql_fence(
    "SELECT 1;"
) == ("SELECT 1;", False)

assert unwrap_single_sql_fence(
    "Here is SQL:\n```sql\nSELECT 1;\n```"
)[1] is False

assert unwrap_single_sql_fence(
    "```sql\nSELECT 1;\n```\nExtra text"
)[1] is False

print("Strict SQL-envelope normalizer validated.")

Strict SQL-envelope normalizer validated.


In [79]:
record_by_id = {
    row["question_id"]: row
    for row in records
    if row["db_name"] == "bird"
    and row["db_id"] == "financial"
}

normalized_guardrail_results = []

for original_result in guardrail_results:
    result = dict(original_result)

    raw_sql = result["generated_sql"]
    normalized_sql, was_unwrapped = (
        unwrap_single_sql_fence(raw_sql)
    )

    example = record_by_id[result["question_id"]]

    reference_sql, rules = make_reference_compatible(
        example["SQL"]
    )

    comparison = compare_sql_results(
        conn=conn,
        reference_sql=reference_sql,
        candidate_sql=normalized_sql,
    )

    result["raw_generated_sql"] = raw_sql
    result["generated_sql"] = normalized_sql
    result["output_envelope_normalized"] = was_unwrapped
    result["passed"] = comparison.equivalent
    result["reason"] = comparison.reason
    result["expected_row_count"] = (
        comparison.expected_row_count
    )
    result["candidate_row_count"] = (
        comparison.candidate_row_count
    )
    result["reference_compatibility_rules"] = rules

    normalized_guardrail_results.append(result)

In [80]:
normalized_passed = sum(
    result["passed"]
    for result in normalized_guardrail_results
)

normalized_executable = sum(
    not result["reason"].startswith(
        "candidate execution failed"
    )
    for result in normalized_guardrail_results
)

unwrapped_ids = [
    result["question_id"]
    for result in normalized_guardrail_results
    if result["output_envelope_normalized"]
]

print("Unwrapped question IDs:", unwrapped_ids)
print(
    f"Accuracy: {normalized_passed}/12 "
    f"({normalized_passed / 12:.1%})"
)
print(
    f"Executable: {normalized_executable}/12 "
    f"({normalized_executable / 12:.1%})"
)

for result in normalized_guardrail_results:
    if result["output_envelope_normalized"]:
        print(
            f"ID {result['question_id']:>3}: "
            f"{'PASS' if result['passed'] else 'FAIL'} — "
            f"{result['reason']}"
        )

Unwrapped question IDs: [14, 61, 89, 105]
Accuracy: 4/12 (33.3%)
Executable: 8/12 (66.7%)
ID  14: FAIL — column count differs: 1 != 2
ID  61: FAIL — unordered results differ
ID  89: FAIL — candidate execution failed: no such column: account.client_id
ID 105: FAIL — candidate execution failed: ambiguous column name: client_id


In [81]:
normalized_run_path = Path(
    "../data/runs/"
    "ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json"
).resolve()

normalized_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": (
        "compact_guardrails_plus_strict_output_unwrap"
    ),
    "source_run": guardrail_run_path.name,
    "context_sha256": EXPECTED_COMPACT_CONTEXT_SHA256,
    "instructions_sha256": EXPECTED_GUARDRAIL_HASH,
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "output_envelope_normalization": (
        "unwrap exactly one surrounding SQL Markdown fence"
    ),
    "model_requests_made": 0,
    "results": normalized_guardrail_results,
}

normalized_run_path.write_text(
    json.dumps(normalized_run, indent=2),
    encoding="utf-8",
)

normalized_run_sha256 = hashlib.sha256(
    normalized_run_path.read_bytes()
).hexdigest()

print("Saved:", normalized_run_path)
print("Derived-run SHA-256:", normalized_run_sha256)
print("No model request made.")

Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json
Derived-run SHA-256: 3432a0d565155a5ae2774c08578882a3f19cb215f924a35e1b804d8eb94e12a5
No model request made.


In [82]:
repair_instructions = (
    """You repair a failed SQLite query.

Return exactly one executable SQLite query.
Preserve the original question's intended answer.
Use only the supplied tables, columns, relationships, meanings, and values.
Fix the execution error without adding unrelated logic.
Do not include Markdown, code fences, comments, or explanations."""
    + "\n\n"
    + sql_construction_guardrails
)

repair_user_template = """Schema and domain context:
{schema_context}

Original question:
{question}

Failed SQL:
{failed_sql}

SQLite execution error:
{execution_error}

Return the repaired SQLite query only."""

In [83]:
repair_contract = {
    "instructions": repair_instructions,
    "user_template": repair_user_template,
    "trigger": "candidate execution failure only",
    "maximum_repairs": 1,
    "reference_sql_available": False,
    "expected_results_available": False,
    "output_normalization": (
        "strict single SQL-fence unwrapping"
    ),
}

repair_contract_json = json.dumps(
    repair_contract,
    sort_keys=True,
    separators=(",", ":"),
)

repair_contract_sha256 = hashlib.sha256(
    repair_contract_json.encode("utf-8")
).hexdigest()

print("Repair-contract SHA-256:", repair_contract_sha256)
print(
    "Repair instruction characters:",
    len(repair_instructions),
)
print(
    "Repair template characters:",
    len(repair_user_template),
)

Repair-contract SHA-256: 64837f617470bc7eb388e9d429f8fd2cf78abee177249bda2730ff586d326e24
Repair instruction characters: 1391
Repair template characters: 183


In [86]:
repair_diagnostic_question_id = 11

repair_failed_result = next(
    result
    for result in normalized_guardrail_results
    if result["question_id"] == repair_diagnostic_question_id
)

assert repair_failed_result["passed"] is False
assert repair_failed_result["reason"].startswith(
    "candidate execution failed:"
)

repair_example = record_by_id[
    repair_diagnostic_question_id
]

execution_error = repair_failed_result[
    "reason"
].removeprefix(
    "candidate execution failed:"
).strip()

repair_user_input = repair_user_template.format(
    schema_context=compact_domain_context,
    question=repair_example["question"],
    failed_sql=repair_failed_result["generated_sql"],
    execution_error=execution_error,
)

repair_input_sha256 = hashlib.sha256(
    repair_user_input.encode("utf-8")
).hexdigest()

print("Question ID:", repair_diagnostic_question_id)
print("Failed SQL:")
print(repair_failed_result["generated_sql"])
print("SQLite error:", execution_error)
print("Repair-input SHA-256:", repair_input_sha256)

Question ID: 11
Failed SQL:
SELECT account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;
SQLite error: ambiguous column name: account_id
Repair-input SHA-256: d172714515d0e4859a07947d6658a155841ffa2718c9aec2d57082519179be68


In [88]:
repair_response = client.chat.completions.create(
    model=slm_deployment_name,
    messages=[
        {
            "role": "system",
            "content": repair_instructions,
        },
        {
            "role": "user",
            "content": repair_user_input,
        },
    ],
    max_tokens=slm_config["max_tokens"],
)

repair_raw_sql = (
    repair_response.choices[0].message.content or ""
).strip()

repair_sql, repair_was_unwrapped = (
    unwrap_single_sql_fence(repair_raw_sql)
)

print("Response ID:", repair_response.id)
print("Response model:", repair_response.model)
print(
    "Finish reason:",
    repair_response.choices[0].finish_reason,
)
print("\nRaw repair output:")
print(repair_raw_sql)
print("\nNormalized repair SQL:")
print(repair_sql)
print(
    "\nOutput envelope normalized:",
    repair_was_unwrapped,
)
print("\nUsage:")
print(repair_response.usage)

Response ID: ad63284a85fe452bb06ee0fdd83ea9f9
Response model: ministral-3b-2410
Finish reason: stop

Raw repair output:
SELECT account.account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;

Normalized repair SQL:
SELECT account.account_id
FROM account
JOIN loan ON account.account_id = loan.account_id
WHERE loan.duration > 12
AND account.date = '1993-01-01'
ORDER BY loan.amount DESC
LIMIT 1;

Output envelope normalized: False

Usage:
CompletionUsage(completion_tokens=56, prompt_tokens=1470, total_tokens=1526, completion_tokens_details=None, prompt_tokens_details=None, audio_prompt_tokens=0)


In [90]:
repair_reference_sql, repair_applied_rules = (
    make_reference_compatible(
        repair_example["SQL"]
    )
)

repair_comparison = compare_sql_results(
    conn=conn,
    reference_sql=repair_reference_sql,
    candidate_sql=repair_sql,
)

print("Question ID:", repair_diagnostic_question_id)
print(
    "Reference-only compatibility rules:",
    repair_applied_rules,
)
print("Passed:", repair_comparison.equivalent)
print("Reason:", repair_comparison.reason)
print(
    "Expected row count:",
    repair_comparison.expected_row_count,
)
print(
    "Candidate row count:",
    repair_comparison.candidate_row_count,
)

Question ID: 11
Reference-only compatibility rules: ['sqlite_four_digit_year']
Passed: False
Reason: ordered results differ
Expected row count: 1
Candidate row count: 0


In [91]:
def response_usage_dict(response):
    return (
        response.usage.model_dump()
        if response.usage is not None
        else {}
    )


def apply_repair_result(
    original_result,
    *,
    response,
    raw_repair_sql,
    repaired_sql,
    was_unwrapped,
    input_sha256,
    comparison,
    applied_rules,
):
    result = dict(original_result)

    result.update({
        "pre_repair_sql": original_result["generated_sql"],
        "pre_repair_reason": original_result["reason"],
        "repair_attempted": True,
        "repair_input_sha256": input_sha256,
        "repair_response_id": response.id,
        "repair_response_model": response.model,
        "repair_finish_reason": (
            response.choices[0].finish_reason
        ),
        "raw_repair_output": raw_repair_sql,
        "repair_envelope_normalized": was_unwrapped,
        "repair_usage": response_usage_dict(response),
        "repair_api_error": None,
        "generated_sql": repaired_sql,
        "passed": comparison.equivalent,
        "reason": comparison.reason,
        "expected_row_count": comparison.expected_row_count,
        "candidate_row_count": comparison.candidate_row_count,
        "reference_compatibility_rules": applied_rules,
    })

    return result


repaired_results_by_id = {}

for original_result in normalized_guardrail_results:
    result = dict(original_result)
    result["repair_attempted"] = False
    result["repair_usage"] = {}
    result["repair_api_error"] = None

    repaired_results_by_id[
        result["question_id"]
    ] = result


repaired_results_by_id[11] = apply_repair_result(
    repaired_results_by_id[11],
    response=repair_response,
    raw_repair_sql=repair_raw_sql,
    repaired_sql=repair_sql,
    was_unwrapped=repair_was_unwrapped,
    input_sha256=repair_input_sha256,
    comparison=repair_comparison,
    applied_rules=repair_applied_rules,
)

In [92]:
repair_run_path = Path(
    "../data/runs/"
    "ministral_3b_compact_guardrails_one_repair_working_dev_v1.json"
).resolve()

repair_run = {
    "run_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": (
        "compact_guardrails_strict_unwrap_plus_one_execution_repair"
    ),
    "source_run": normalized_run_path.name,
    "context_sha256": EXPECTED_COMPACT_CONTEXT_SHA256,
    "generation_instructions_sha256": (
        EXPECTED_GUARDRAIL_HASH
    ),
    "repair_contract_sha256": (
        "64837f617470bc7eb388e9d429f8fd2cf78abee177249bda2730ff586d326e24"
    ),
    "repair_trigger": "candidate execution failure only",
    "maximum_repairs_per_question": 1,
    "reference_sql_available_to_repair": False,
    "expected_results_available_to_repair": False,
    "results": sorted(
        repaired_results_by_id.values(),
        key=lambda result: result["question_id"],
    ),
}


def save_repair_run():
    repair_run["results"] = sorted(
        repaired_results_by_id.values(),
        key=lambda result: result["question_id"],
    )

    repair_run_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = repair_run_path.with_suffix(
        ".json.tmp"
    )

    temporary_path.write_text(
        json.dumps(repair_run, indent=2),
        encoding="utf-8",
    )

    temporary_path.replace(repair_run_path)


save_repair_run()

In [93]:
pending_repair_ids = [
    question_id
    for question_id, result in sorted(
        repaired_results_by_id.items()
    )
    if (
        question_id != 11
        and result["reason"].startswith(
            "candidate execution failed:"
        )
    )
]

print("Pending repair IDs:", pending_repair_ids)

assert pending_repair_ids == [81, 89, 105]

Pending repair IDs: [81, 89, 105]


In [94]:
for index, question_id in enumerate(
    pending_repair_ids,
    start=1,
):
    original_result = repaired_results_by_id[
        question_id
    ]
    example = record_by_id[question_id]

    execution_error = original_result[
        "reason"
    ].removeprefix(
        "candidate execution failed:"
    ).strip()

    repair_input = repair_user_template.format(
        schema_context=compact_domain_context,
        question=example["question"],
        failed_sql=original_result["generated_sql"],
        execution_error=execution_error,
    )

    input_sha256 = hashlib.sha256(
        repair_input.encode("utf-8")
    ).hexdigest()

    try:
        response = client.chat.completions.create(
            model=slm_deployment_name,
            messages=[
                {
                    "role": "system",
                    "content": repair_instructions,
                },
                {
                    "role": "user",
                    "content": repair_input,
                },
            ],
            max_tokens=slm_config["max_tokens"],
        )

        raw_repair_sql = (
            response.choices[0].message.content or ""
        ).strip()

        repaired_sql, was_unwrapped = (
            unwrap_single_sql_fence(raw_repair_sql)
        )

        reference_sql, rules = (
            make_reference_compatible(
                example["SQL"]
            )
        )

        comparison = compare_sql_results(
            conn=conn,
            reference_sql=reference_sql,
            candidate_sql=repaired_sql,
        )

        repaired_result = apply_repair_result(
            original_result,
            response=response,
            raw_repair_sql=raw_repair_sql,
            repaired_sql=repaired_sql,
            was_unwrapped=was_unwrapped,
            input_sha256=input_sha256,
            comparison=comparison,
            applied_rules=rules,
        )

    except Exception as error:
        repaired_result = dict(original_result)
        repaired_result.update({
            "repair_attempted": True,
            "repair_input_sha256": input_sha256,
            "repair_usage": {},
            "repair_api_error": (
                f"{type(error).__name__}: {error}"
            ),
        })

    repaired_results_by_id[
        question_id
    ] = repaired_result

    save_repair_run()

    print(
        f"[{index:02d}/{len(pending_repair_ids):02d}] "
        f"ID {question_id:>3}: "
        f"{'PASS' if repaired_result['passed'] else 'FAIL'} — "
        f"{repaired_result['reason']}"
    )

[01/03] ID  81: FAIL — column count differs: 1 != 2
[02/03] ID  89: FAIL — candidate execution failed: no such column: account.client_id
[03/03] ID 105: FAIL — candidate execution failed: no such column: district.A2


In [95]:
repaired_results = sorted(
    repaired_results_by_id.values(),
    key=lambda result: result["question_id"],
)

repaired_passed = sum(
    result["passed"]
    for result in repaired_results
)

repaired_executable = sum(
    not result["reason"].startswith(
        "candidate execution failed"
    )
    for result in repaired_results
)

repair_tokens = {
    token_name: sum(
        result.get("repair_usage", {}).get(
            token_name,
            0,
        )
        for result in repaired_results
    )
    for token_name in (
        "prompt_tokens",
        "completion_tokens",
        "total_tokens",
    )
}

print(
    f"Accuracy: {repaired_passed}/12 "
    f"({repaired_passed / 12:.1%})"
)
print(
    f"Executable: {repaired_executable}/12 "
    f"({repaired_executable / 12:.1%})"
)
print("Additional repair-token usage:", repair_tokens)

for result in repaired_results:
    if result["repair_attempted"]:
        print(
            f"ID {result['question_id']:>3}: "
            f"{result['pre_repair_reason']} "
            f"-> {result['reason']} "
            f"| PASS={result['passed']}"
        )

repair_run_sha256 = hashlib.sha256(
    repair_run_path.read_bytes()
).hexdigest()

print("Saved:", repair_run_path)
print("Repair-run SHA-256:", repair_run_sha256)

Accuracy: 4/12 (33.3%)
Executable: 10/12 (83.3%)
Additional repair-token usage: {'prompt_tokens': 6043, 'completion_tokens': 452, 'total_tokens': 6495}
ID  11: candidate execution failed: ambiguous column name: account_id -> ordered results differ | PASS=False
ID  81: candidate execution failed: no such column: account.account_id -> column count differs: 1 != 2 | PASS=False
ID  89: candidate execution failed: no such column: account.client_id -> candidate execution failed: no such column: account.client_id | PASS=False
ID 105: candidate execution failed: ambiguous column name: client_id -> candidate execution failed: no such column: district.A2 | PASS=False
Saved: C:\Users\shchitt\Downloads\Projects\domain-adapt\data\runs\ministral_3b_compact_guardrails_one_repair_working_dev_v1.json
Repair-run SHA-256: efaf3df62e66cbfcea42f128fa22c32700dc700714b5028f5abd47b4e8a6a6aa
